# Arquivo de carregamento dos arquivos .csv para .parquet

In [ ]:
# ==============================================================================================================
# ESTE É O ARQUIVO RESPONSÁVEL POR TRANSFORMAR OS ARQUIVOS CSV EM ARQUIVOS .PARQUET, PARA A MELHOR ANÁLISE DELES
# ==============================================================================================================

import pandas as pd
import os
import glob
import gc
from google.colab import drive
import time

# =============================================
# 1. MONTAR O GOOGLE DRIVE
# =============================================
drive.mount('/content/drive')

# =============================================
# 2. CONFIGURAR OS CAMINHOS (AJUSTE AQUI!)
# =============================================

PASTA_ORIGEM = '/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor'
PASTA_DESTINO = PASTA_ORIGEM + " " + '/Parquet/'

# Criar a pasta de destino se não existir
os.makedirs(PASTA_DESTINO, exist_ok=True)

# =============================================
# 3. CONFIGURAÇÕES DE LEITURA DO CSV
# =============================================

ENCODING = 'latin-1'
SEPARADOR = ';'
DECIMAL = ','

# =============================================
# 4. LOOP PRINCIPAL (2013 até 2025)
# =============================================

anos = list(range(2013, 2026))

for ano in anos:
    print(f"\n{'='*50}")
    print(f"📂 Processando ano: {ano}")

    # Caminho da pasta do ano
    pasta_ano = os.path.join(PASTA_ORIGEM, str(ano))

    if not os.path.exists(pasta_ano):
        print(f"⚠️  Pasta {pasta_ano} não encontrada. Pulando...")
        continue

    # =============================================
    # 4a. PROCURAR A SUBPASTA "CSV + ANO"
    # =============================================

    # Padrões possíveis: "CSV 2013", "CSV_2013", "CSV-2013", "csv2013" etc.
    # Vamos listar todas as subpastas dentro da pasta do ano
    subpastas = [d for d in os.listdir(pasta_ano)
                 if os.path.isdir(os.path.join(pasta_ano, d))]

    # Filtrar as que contêm "CSV" (case insensitive) e o ano
    pasta_csv = None
    for sub in subpastas:
        # Verifica se "csv" está no nome (ignorando maiúsculas/minúsculas)
        if 'csv' in sub.lower() and str(ano) in sub:
            pasta_csv = sub
            break

    # Se não encontrou com "CSV", tenta pegar qualquer subpasta que tenha o ano
    if pasta_csv is None:
        for sub in subpastas:
            if str(ano) in sub:
                pasta_csv = sub
                break

    if pasta_csv is None:
        print(f"❌ Nenhuma subpasta com 'CSV' ou com o ano {ano} encontrada em {pasta_ano}. Pulando...")
        continue

    caminho_csv = os.path.join(pasta_ano, pasta_csv)
    print(f"   📁 Subpasta encontrada: {pasta_csv}")

    # =============================================
    # 4b. PROCURAR O ARQUIVO CSV DENTRO DA SUBPASTA
    # =============================================

    # Lista todos os arquivos .csv na subpasta
    lista_csvs = glob.glob(os.path.join(caminho_csv, "*.csv"))

    if not lista_csvs:
        print(f"❌ Nenhum arquivo CSV encontrado em {caminho_csv}. Pulando...")
        continue

    # Pega o primeiro CSV (ou você pode adaptar para pegar um específico)
    arquivo_csv = lista_csvs[0]
    print(f"   📄 Arquivo encontrado: {os.path.basename(arquivo_csv)}")

    # =============================================
    # 4c. LER O CSV
    # =============================================
    try:
        print(f"   ⏳ Lendo CSV... (pode levar alguns segundos)")
        df = pd.read_csv(
            arquivo_csv,
            encoding=ENCODING,
            sep=SEPARADOR,
            decimal=DECIMAL,
            on_bad_lines='warn',
            low_memory=False
        )

        print(f"   ✅ Leitura concluída! Linhas: {len(df)}, Colunas: {len(df.columns)}")
        print(f"   📋 Primeiras colunas: {df.columns[:5].tolist()}")

    except Exception as e:
        print(f"   ❌ ERRO ao ler o CSV: {e}")
        continue

    # =============================================
    # 4d. SALVAR COMO PARQUET
    # =============================================

    nome_base = os.path.splitext(os.path.basename(arquivo_csv))[0]
    arquivo_parquet = os.path.join(PASTA_DESTINO, f"{nome_base}.parquet")

    try:
        print(f"   ⏳ Convertendo e salvando como Parquet...")
        df.to_parquet(
            arquivo_parquet,
            compression='snappy',
            index=False
        )

        tamanho_mb = os.path.getsize(arquivo_parquet) / (1024 * 1024)
        print(f"   ✅ Salvo com sucesso! Tamanho: {tamanho_mb:.2f} MB")

    except Exception as e:
        print(f"   ❌ ERRO ao salvar Parquet: {e}")
        continue

    # =============================================
    # 4e. LIMPAR MEMÓRIA
    # =============================================
    del df
    gc.collect()
    print(f"   🧹 Memória liberada para o próximo ano.")

print(f"\n{'='*50}")
print("🎉 PROCESSAMENTO CONCLUÍDO! Todos os anos foram convertidos.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📂 Processando ano: 2013
   📁 Subpasta encontrada: CSV 2013
   📄 Arquivo encontrado: SICOR_OPERACAO_BASICA_ESTADO_2013.csv
   ⏳ Lendo CSV... (pode levar alguns segundos)
   ✅ Leitura concluída! Linhas: 2804692, Colunas: 47
   📋 Primeiras colunas: ['#REF_BACEN', 'NU_ORDEM', 'CNPJ_IF', 'DT_EMISSAO', 'DT_VENCIMENTO']
   ⏳ Convertendo e salvando como Parquet...
   ✅ Salvo com sucesso! Tamanho: 69.68 MB
   🧹 Memória liberada para o próximo ano.

📂 Processando ano: 2014
   📁 Subpasta encontrada: CSV 2014
   📄 Arquivo encontrado: SICOR_OPERACAO_BASICA_ESTADO_2014.csv
   ⏳ Lendo CSV... (pode levar alguns segundos)
   ✅ Leitura concluída! Linhas: 2584654, Colunas: 47
   📋 Primeiras colunas: ['#REF_BACEN', 'NU_ORDEM', 'CNPJ_IF', 'DT_EMISSAO', 'DT_VENCIMENTO']
   ⏳ Convertendo e salvando como Parquet...
   ✅ Salvo com sucesso! Tamanho: 67.00 MB
   🧹 Memória liberada par

# Arquivo de Exploração dos .parquet - Total de Crédito Rural usado em bilhões de reais

In [ ]:
# =============================================
# INSTALAÇÃO (se necessário)
# =============================================
!pip install plotly -U

# =============================================
# IMPORTS
# =============================================
from google.colab import drive
drive.mount('/content/drive')

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import glob
import os
import re

# =============================================
# CONFIGURAÇÃO - PASTA DOS ARQUIVOS
# =============================================
caminho_pasta = '/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet'
arquivos = sorted(glob.glob(os.path.join(caminho_pasta, '*.parquet')))
print(f"Encontrados {len(arquivos)} arquivos.")

arquivos_filtrados = arquivos
print(f"Processando: {[os.path.basename(a) for a in arquivos_filtrados]}")

# =============================================
# FUNÇÃO PARA LIMPAR COLUNAS MONETÁRIAS (CORRIGIDA)
# =============================================
def tratar_coluna_br(lf, coluna):
    """Converte a coluna para float e força a divisão por 100 (ajuste de centavos)."""
    dtype = lf.schema[coluna]

    # Checagem agnóstica de versão para lidar com String ou Utf8 no Polars
    if str(dtype) in ('String', 'Utf8'):
        expr = (pl.col(coluna)
                .str.replace_all(r'\.', '')
                .str.replace(',', '.')
                .cast(pl.Float64, strict=False) / 100.0)
    else:
        expr = pl.col(coluna).cast(pl.Float64, strict=False) / 100.0

    return expr.alias(coluna)

# =============================================
# 1. PROCESSAR CADA ARQUIVO E EXTRAIR AGREGADOS
# =============================================
agregados = []

for arq in arquivos_filtrados:
    nome = os.path.basename(arq)
    ano_match = re.search(r'(\d{4})', nome)
    ano = int(ano_match.group(1)) if ano_match else None
    if ano is None:
        continue

    print(f"Processando {nome} (ano {ano})...")

    lf = pl.scan_parquet(arq)

    # --- Limpeza nas colunas numéricas ---
    if 'VL_PARC_CREDITO' in lf.columns:
        lf = lf.with_columns(tratar_coluna_br(lf, 'VL_PARC_CREDITO'))
    if 'VL_JUROS' in lf.columns:
        lf = lf.with_columns(tratar_coluna_br(lf, 'VL_JUROS'))

    # --- 1a. Métricas anuais consolidadas ---
    expressoes_selecao = [
        pl.sum('VL_PARC_CREDITO').alias('total_credito'),
        pl.count().alias('total_operacoes')
    ]
    if 'VL_JUROS' in lf.columns:
        expressoes_selecao.append(pl.mean('VL_JUROS').alias('juros_medio'))

    resultado_global = lf.select(expressoes_selecao).collect()

    total_credito = resultado_global['total_credito'][0]
    total_operacoes = resultado_global['total_operacoes'][0]
    juros_medio = resultado_global['juros_medio'][0] if 'VL_JUROS' in lf.columns else None

    # --- 1b. Agregado por ESTADO (CD_ESTADO) ---
    if 'CD_ESTADO' in lf.columns:
        expressoes_estado = [
            pl.sum('VL_PARC_CREDITO').alias('TOTAL_CREDITO'),
            pl.count().alias('OPERACOES')
        ]
        if 'VL_JUROS' in lf.columns:
            expressoes_estado.append(pl.mean('VL_JUROS').alias('JUROS_MEDIO'))

        agg_estado = (lf
                      .group_by('CD_ESTADO')
                      .agg(expressoes_estado)
                      .collect()
                      .to_pandas())
        agg_estado['ANO'] = ano
    else:
        agg_estado = None

    # --- 1c. Agregado por PROGRAMA (CD_PROGRAMA) ---
    if 'CD_PROGRAMA' in lf.columns:
        agg_programa = (lf
                        .group_by('CD_PROGRAMA')
                        .agg([
                            pl.sum('VL_PARC_CREDITO').alias('TOTAL_CREDITO'),
                            pl.count().alias('OPERACOES')
                        ])
                        .sort('TOTAL_CREDITO', descending=True)
                        .head(10)
                        .collect()
                        .to_pandas())
        agg_programa['ANO'] = ano
    else:
        agg_programa = None

    agregados.append({
        'ano': ano,
        'total_credito': total_credito,
        'total_operacoes': total_operacoes,
        'juros_medio': juros_medio,
        'agg_estado': agg_estado,
        'agg_programa': agg_programa
    })

# =============================================
# 2. GRÁFICOS COM PLOTLY (FORMATADOS EM BILHÕES)
# =============================================

# --- 2.1. Evolução do crédito total por ano ---
df_evolucao = pl.DataFrame({
    'ANO': [a['ano'] for a in agregados],
    'TOTAL_CREDITO': [a['total_credito'] for a in agregados],
    'OPERACOES': [a['total_operacoes'] for a in agregados]
}).sort('ANO').to_pandas()

# OTIMIZAÇÃO VISUAL: Divide por 1 bilhão para apresentar "Bi" corretamente no gráfico
df_evolucao['TOTAL_CREDITO_BI'] = df_evolucao['TOTAL_CREDITO'] / 1e9

fig1 = px.bar(df_evolucao, x='ANO', y='TOTAL_CREDITO_BI',
              title='💰 Total de Crédito por Ano (Bilhões de R$)',
              color='ANO',
              color_continuous_scale='Viridis')
fig1.update_traces(texttemplate='%{y:.0f} Bi', textposition='outside')
fig1.update_layout(yaxis_title="Total de Crédito (Bilhões)")
fig1.show()

# --- 2.2. Top 10 estados (Ano mais recente) ---
ano_mais_recente = max([a['ano'] for a in agregados])
agg_estado_mais_recente = next(a['agg_estado'] for a in agregados if a['ano'] == ano_mais_recente)
if agg_estado_mais_recente is not None:
    top_estados = agg_estado_mais_recente.sort_values('TOTAL_CREDITO', ascending=False).head(10).copy()

    # Aplicando a mesma conversão para Bilhões
    top_estados['TOTAL_CREDITO_BI'] = top_estados['TOTAL_CREDITO'] / 1e9

    fig2 = px.bar(top_estados, x='CD_ESTADO', y='TOTAL_CREDITO_BI',
                  title=f'🏆 Top 10 Estados com Maior Crédito em {ano_mais_recente} (Bilhões de R$)',
                  color='TOTAL_CREDITO_BI',
                  color_continuous_scale='Blues')
    fig2.update_traces(texttemplate='%{y:.1f} Bi', textposition='outside')
    fig2.update_layout(yaxis_title="Total de Crédito (Bilhões)")
    fig2.show()

# --- 2.3. Distribuição de programas (Ano mais recente) ---
agg_programa_mais_recente = next(a['agg_programa'] for a in agregados if a['ano'] == ano_mais_recente)
if agg_programa_mais_recente is not None:
    fig3 = px.pie(agg_programa_mais_recente.head(10), values='OPERACOES', names='CD_PROGRAMA',
                  title=f'📊 Distribuição dos Programas de Crédito em {ano_mais_recente}',
                  hole=0.3)
    fig3.show()

# --- 2.4. Comparação de juros médios por estado ---
if len(agregados) >= 2:
    lista_dfs_juros = []
    for a in agregados:
        if a['agg_estado'] is not None and 'JUROS_MEDIO' in a['agg_estado'].columns:
            lista_dfs_juros.append(pl.from_pandas(a['agg_estado']).select(['CD_ESTADO', 'JUROS_MEDIO', 'ANO']))

    if lista_dfs_juros:
        df_juros_estado = pl.concat(lista_dfs_juros).to_pandas()
        top_estados_lista = agg_estado_mais_recente.nlargest(10, 'OPERACOES')['CD_ESTADO'].tolist()
        df_juros_filtrado = df_juros_estado[df_juros_estado['CD_ESTADO'].isin(top_estados_lista)]

        fig4 = px.bar(df_juros_filtrado, x='CD_ESTADO', y='JUROS_MEDIO', color='ANO',
                      barmode='group',
                      title='📉 Taxa de Juros Média por Estado (comparação)',
                      labels={'JUROS_MEDIO': 'Juros Médio (%)'})
        fig4.show()

# --- 2.5. Tabela resumo por ano (formatada) ---
resumo = df_evolucao.copy()
resumo['TOTAL_CREDITO'] = resumo['TOTAL_CREDITO'].apply(lambda x: f'R$ {x:,.2f}' if x is not None else 'N/A')
resumo['OPERACOES'] = resumo['OPERACOES'].apply(lambda x: f'{x:,.0f}' if x is not None else 'N/A')
print("\n📋 RESUMO POR ANO:")
print(resumo[['ANO', 'TOTAL_CREDITO', 'OPERACOES']].to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Encontrados 13 arquivos.
Processando: ['SICOR_OPERACAO_BASICA_ESTADO_2013.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2014.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2015.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2016.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2017.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2018.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2019.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2020.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2021.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2022.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2023.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2024.parquet', 'SICOR_OPERACAO_BASICA_ESTADO_2025.parquet']
Processando SICOR_OPERACAO_BASICA_ESTADO_2013.parquet (ano 2013)...


/tmp/ipykernel_5427/445075956.py:64: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_PARC_CREDITO' in lf.columns:
/tmp/ipykernel_5427/445075956.py:34: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dtype = lf.schema[coluna]
/tmp/ipykernel_5427/445075956.py:66: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_JUROS' in lf.columns:
/tmp/ipykernel_5427/445075956.py:72: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias('total_

Processando SICOR_OPERACAO_BASICA_ESTADO_2014.parquet (ano 2014)...


/tmp/ipykernel_5427/445075956.py:64: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_PARC_CREDITO' in lf.columns:
/tmp/ipykernel_5427/445075956.py:34: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dtype = lf.schema[coluna]
/tmp/ipykernel_5427/445075956.py:66: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_JUROS' in lf.columns:
/tmp/ipykernel_5427/445075956.py:72: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias('total_

Processando SICOR_OPERACAO_BASICA_ESTADO_2015.parquet (ano 2015)...


...

Processando SICOR_OPERACAO_BASICA_ESTADO_2024.parquet (ano 2024)...


/tmp/ipykernel_5427/445075956.py:64: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_PARC_CREDITO' in lf.columns:
/tmp/ipykernel_5427/445075956.py:34: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dtype = lf.schema[coluna]
/tmp/ipykernel_5427/445075956.py:66: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_JUROS' in lf.columns:
/tmp/ipykernel_5427/445075956.py:72: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias('total_

Processando SICOR_OPERACAO_BASICA_ESTADO_2025.parquet (ano 2025)...


/tmp/ipykernel_5427/445075956.py:64: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_PARC_CREDITO' in lf.columns:
/tmp/ipykernel_5427/445075956.py:34: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dtype = lf.schema[coluna]
/tmp/ipykernel_5427/445075956.py:66: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'VL_JUROS' in lf.columns:
/tmp/ipykernel_5427/445075956.py:72: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias('total_


📋 RESUMO POR ANO:
 ANO         TOTAL_CREDITO OPERACOES
2013 R$ 139,386,117,087.24 2,804,692
2014 R$ 164,430,998,668.78 2,584,654
2015 R$ 154,146,000,570.27 2,330,156
2016 R$ 117,624,099,741.70 1,576,888
2017 R$ 167,016,509,368.07 2,109,447
2018 R$ 180,884,316,899.71 2,033,300
2019 R$ 178,947,752,584.10 1,860,900
2020 R$ 206,999,111,958.64 1,924,778
2021 R$ 294,987,494,742.29 1,989,446
2022 R$ 361,983,106,759.88 1,990,945
2023 R$ 406,395,533,265.62 2,129,134
2024 R$ 379,777,789,269.24 2,328,948
2025 R$ 358,138,962,543.60 2,442,939


# Análise das Colunas Para o Dataset Sintético - SICOR

In [ ]:
# =============================================
# 0. IMPORTS E CONFIGURAÇÃO
# =============================================
from google.colab import drive
drive.mount('/content/drive')

import polars as pl
import pyarrow.parquet as pq
import glob
import os
import re
import pandas as pd

# =============================================
# 1. APONTE PARA A PASTA DOS ARQUIVOS
# =============================================
caminho_pasta = '/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet'  # <--- MUDE AQUI
arquivos = sorted(glob.glob(os.path.join(caminho_pasta, '*.parquet')))

if not arquivos:
    print("❌ Nenhum arquivo .parquet encontrado!")
    exit()

print(f"🔍 Encontrei {len(arquivos)} arquivo(s):")
for a in arquivos:
    print(f"   - {os.path.basename(a)}")

# =============================================
# 2. COLUNAS DE INTERESSE (QUE VOCÊ QUER USAR)
# =============================================
colunas_interesse = [
    # Mutuários
    'CD_TIPO_BENEFICIARIO', 'CD_SEXO', 'CD_DAP',
    # Operações
    'VL_AREA_FINANC', 'VL_PARC_CREDITO', 'VL_REC_PROPRIO', 'VL_PREV_PROD',
    'CD_PROGRAMA', 'CD_SUBPROGRAMA', 'CD_IBGE_MUNICIPIO',
    'DT_INICIO_PLANTIO', 'DT_FIM_PLANTIO',
    'DT_INICIO_COLHEITA', 'DT_FIM_COLHEITA',
    'CD_ZONEAMENTO', 'VL_JUROS',
    'VL_PREC_CUSTO_EFET_TOTAL', 'VL_RECEITA_BRUTA_ESPERADA',
    # Parcelas (se tiver)
    # (Adicione aqui as colunas específicas de parcelas se souber os nomes)
]

# =============================================
# 3. FUNÇÃO PARA EXTRAIR METADADOS E CONTAR NULOS
# =============================================
def analisar_arquivo(caminho, colunas_alvo):
    nome = os.path.basename(caminho)

    # Total de linhas (instantâneo)
    try:
        meta = pq.ParquetFile(caminho)
        total_linhas = meta.metadata.num_rows
    except:
        total_linhas = None

    # Schema (instantâneo)
    schema = pl.scan_parquet(caminho).schema
    colunas_existentes = list(schema.keys())

    # Verifica quais colunas de interesse existem neste arquivo
    colunas_presentes = [col for col in colunas_alvo if col in colunas_existentes]

    # Se não houver nenhuma coluna de interesse, retorna apenas metadados
    if not colunas_presentes:
        return {
            'arquivo': nome,
            'total_linhas': total_linhas,
            'colunas_presentes': colunas_existentes,
            'colunas_interesse_presentes': [],
            'nulos': {}
        }

    # Contagem de nulos para as colunas de interesse (usando LazyFrame)
    lf = pl.scan_parquet(caminho)
    df_nulos = lf.select([pl.col(col).is_null().sum().alias(col) for col in colunas_presentes]).collect()

    nulos_dict = {}
    for col in colunas_presentes:
        nulos_dict[col] = df_nulos[col].item()

    return {
        'arquivo': nome,
        'total_linhas': total_linhas,
        'colunas_presentes': colunas_existentes,
        'colunas_interesse_presentes': colunas_presentes,
        'nulos': nulos_dict
    }

# =============================================
# 4. ANALISAR TODOS OS ARQUIVOS
# =============================================
print("\n⏳ Analisando os arquivos...\n")
resultados = []

for arq in arquivos:
    print(f"   📂 {os.path.basename(arq)}...")
    res = analisar_arquivo(arq, colunas_interesse)
    resultados.append(res)

# =============================================
# 5. EXIBIR RELATÓRIO: MATRIZ DE PRESENÇA DAS COLUNAS
# =============================================
print("\n" + "="*90)
print("📋 MATRIZ DE PRESENÇA DAS COLUNAS DE INTERESSE")
print("="*90)

# Monta uma tabela com os arquivos como linhas e colunas como colunas
df_presenca = pd.DataFrame()
for res in resultados:
    linha = {'Arquivo': res['arquivo']}
    for col in colunas_interesse:
        if col in res['colunas_interesse_presentes']:
            linha[col] = '✅'
        else:
            linha[col] = '❌'
    df_presenca = pd.concat([df_presenca, pd.DataFrame([linha])], ignore_index=True)

print(df_presenca.to_string(index=False))

# =============================================
# 6. RELATÓRIO DE NULOS POR COLUNA (COM STATUS)
# =============================================
print("\n" + "="*90)
print("📊 ANÁLISE DE NULOS PARA CADA COLUNA DE INTERESSE")
print("="*90)

# Função para classificar o percentual de nulos
def classificar_nulos(pct):
    if pct <= 5:
        return '🟢 OK'
    elif pct <= 20:
        return '🟡 Atenção'
    else:
        return '🔴 Crítico'

# Criar um DataFrame com todos os resultados de nulos
dados_nulos = []
for res in resultados:
    for col, nulos in res['nulos'].items():
        total = res['total_linhas']
        pct = (nulos / total * 100) if total else 0
        dados_nulos.append({
            'Arquivo': res['arquivo'],
            'Coluna': col,
            'Total_Linhas': total,
            'Nulos': nulos,
            'Percentual': round(pct, 2),
            'Status': classificar_nulos(pct)
        })

if dados_nulos:
    df_nulos = pd.DataFrame(dados_nulos)
    # Ordena por arquivo e coluna
    df_nulos = df_nulos.sort_values(['Arquivo', 'Coluna'])
    print(df_nulos.to_string(index=False))
else:
    print("Nenhuma coluna de interesse encontrada nos arquivos.")

# =============================================
# 7. RESUMO GERAL: QUAIS COLUNAS ESTÃO "SAUDÁVEIS" PARA USAR?
# =============================================
print("\n" + "="*90)
print("✅ RESUMO FINAL - COLUNAS RECOMENDADAS PARA O DATASET SINTÉTICO")
print("="*90)

# Filtra as colunas com status 'OK' ou 'Atenção' (se tiver poucos nulos)
if dados_nulos:
    df_recomendadas = df_nulos[df_nulos['Status'] != '🔴 Crítico'].copy()
    print(f"\n📌 Colunas com menos de 20% de nulos (podem ser usadas):")
    if not df_recomendadas.empty:
        # Agrupa por coluna para ver em quais arquivos aparece
        resumo_colunas = df_recomendadas.groupby('Coluna').agg({
            'Arquivo': lambda x: ', '.join(x),
            'Percentual': lambda x: f"mín: {x.min()}% | máx: {x.max()}%"
        }).reset_index()
        print(resumo_colunas.to_string(index=False))
    else:
        print("   ⚠️ Nenhuma coluna com menos de 20% de nulos.")

    # Colunas críticas (mais de 20% de nulos) - talvez descartar ou imputar
    df_criticas = df_nulos[df_nulos['Status'] == '🔴 Crítico'].copy()
    if not df_criticas.empty:
        print("\n🔴 Colunas com MAIS de 20% de nulos (use com cuidado ou descarte):")
        print(df_criticas[['Coluna', 'Arquivo', 'Percentual']].to_string(index=False))
else:
    print("Nenhuma coluna de interesse encontrada. Verifique se os nomes estão corretos.")

# =============================================
# 8. (BÔNUS) LISTA COMPLETA DE COLUNAS POR ARQUIVO
# =============================================
print("\n" + "="*90)
print("📂 COLUNAS DISPONÍVEIS EM CADA ARQUIVO (TODAS)")
print("="*90)
for res in resultados:
    print(f"\n📄 {res['arquivo']} ({res['total_linhas']:,} linhas):")
    print(f"   {', '.join(res['colunas_presentes'][:10])}" + ("..." if len(res['colunas_presentes']) > 10 else ""))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Encontrei 13 arquivo(s):
   - SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2014.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2015.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2016.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2017.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2018.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2019.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2020.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2021.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2022.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2023.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2024.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2025.parquet

⏳ Analisando os arquivos...

   📂 SICOR_OPERACAO_BASICA_ESTADO_2013.parquet...


/tmp/ipykernel_23078/712108537.py:59: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = pl.scan_parquet(caminho).schema


   📂 SICOR_OPERACAO_BASICA_ESTADO_2014.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2015.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2016.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2017.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2018.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2019.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2020.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2021.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2022.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2023.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2024.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2025.parquet...

📋 MATRIZ DE PRESENÇA DAS COLUNAS DE INTERESSE
                                  Arquivo CD_TIPO_BENEFICIARIO CD_SEXO CD_DAP VL_AREA_FINANC VL_PARC_CREDITO VL_REC_PROPRIO VL_PREV_PROD CD_PROGRAMA CD_SUBPROGRAMA CD_IBGE_MUNICIPIO DT_INICIO_PLANTIO DT_FIM_PLANTIO DT_INICIO_COLHEITA DT_FIM_COLHEITA CD_ZONEAMENTO VL_JUROS VL_PREC_CUSTO_EFET_TOTAL VL_RECEITA_BRUTA_ESPERADA
SICOR_OPERACAO_BASICA_ESTADO_

# Verificando a Saúde das Colunas de CD_PROGRAMA nos Dados do SICOR

In [ ]:
# =============================================
# 0. IMPORTS E CONFIGURAÇÃO
# =============================================
from google.colab import drive
drive.mount('/content/drive')

import polars as pl
import pyarrow.parquet as pq
import glob
import os
import re
import pandas as pd

# =============================================
# 1. APONTE PARA A PASTA DOS ARQUIVOS PARQUET
# =============================================
caminho_pasta = '/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet'  # <--- MUDE AQUI SE NECESSÁRIO
arquivos = sorted(glob.glob(os.path.join(caminho_pasta, '*.parquet')))

if not arquivos:
    print("❌ Nenhum arquivo .parquet encontrado!")
    exit()

print(f"🔍 Encontrei {len(arquivos)} arquivo(s):")
for a in arquivos:
    print(f"   - {os.path.basename(a)}")

# =============================================
# 2. FUNÇÃO PARA ANALISAR COLUNAS COM "PROGRAMA"
# =============================================
def analisar_colunas_programa(caminho_parquet):
    """
    Retorna um dicionário com:
      - nome do arquivo
      - total de linhas
      - lista de colunas que contêm "programa" (case insensitive)
      - para cada uma: total, nulos, pct_nulos, n_unique, top5
    """
    nome_arq = os.path.basename(caminho_parquet)

    # Obtém total de linhas (via metadados) e schema
    try:
        meta = pq.ParquetFile(caminho_parquet)
        total_linhas = meta.metadata.num_rows
    except:
        total_linhas = None

    # Lê schema com Polars (instantâneo)
    schema = pl.scan_parquet(caminho_parquet).schema
    todas_colunas = list(schema.keys())

    # Filtra colunas que contêm "programa" (ignorando maiúsculas/minúsculas)
    cols_programa = [col for col in todas_colunas if re.search(r'programa', col, re.IGNORECASE)]

    resultado = {
        'arquivo': nome_arq,
        'total_linhas': total_linhas,
        'todas_colunas': todas_colunas,
        'cols_programa': cols_programa,
        'analise': {}  # coluna -> dict com estatísticas
    }

    if not cols_programa:
        return resultado

    # Para cada coluna, calcular nulos e valores únicos usando LazyFrame
    # (fazemos uma única scan para todas as colunas, para eficiência)
    lf = pl.scan_parquet(caminho_parquet)

    # Expressões: contagem de nulos e contagem de distintos
    exprs = []
    for col in cols_programa:
        exprs.append(pl.col(col).is_null().sum().alias(f'{col}_nulos'))
        exprs.append(pl.col(col).n_unique().alias(f'{col}_unicos'))

    # Coleta os agregados (uma única passagem)
    df_aggr = lf.select(exprs).collect()

    # Para cada coluna, extrai os valores e também os top5
    for col in cols_programa:
        nulos = df_aggr[f'{col}_nulos'][0]
        unicos = df_aggr[f'{col}_unicos'][0]
        pct_nulos = (nulos / total_linhas * 100) if total_linhas else 0

        # Top 5 valores mais frequentes (precisa de uma segunda consulta)
        # Para não sobrecarregar, só fazemos se a coluna tiver menos de 1000 valores únicos
        # (senão, pode demorar muito)
        top5 = {}
        if unicos < 1000 and unicos > 0:
            top_series = lf.select(pl.col(col).value_counts()).collect()
            # O resultado é uma struct com 'counts' e 'values'
            # Vamos extrair os 5 primeiros
            if not top_series.is_empty():
                # Estrutura: DataFrame com uma coluna "counts" que é uma struct
                # Melhor: usar value_counts diretamente
                top5_df = lf.select(pl.col(col)).collect().to_pandas()[col].value_counts().head(5)
                top5 = top5_df.to_dict()
        else:
            top5 = {'(many distinct)': unicos}

        resultado['analise'][col] = {
            'total': total_linhas,
            'nulos': nulos,
            'pct_nulos': round(pct_nulos, 2),
            'unicos': unicos,
            'top5': top5
        }

    return resultado

# =============================================
# 3. ANALISAR TODOS OS ARQUIVOS
# =============================================
print("\n⏳ Analisando colunas com 'PROGRAMA' em cada arquivo...\n")
resultados = []

for arq in arquivos:
    print(f"   📂 {os.path.basename(arq)}...")
    res = analisar_colunas_programa(arq)
    resultados.append(res)

# =============================================
# 4. RELATÓRIO POR ARQUIVO (presença das colunas)
# =============================================
print("\n" + "="*90)
print("📋 COLUNAS COM 'PROGRAMA' ENCONTRADAS EM CADA ARQUIVO")
print("="*90)

for res in resultados:
    print(f"\n📄 {res['arquivo']} ({res['total_linhas']:,} linhas):")
    if res['cols_programa']:
        for col in res['cols_programa']:
            analise = res['analise'][col]
            status = '🟢' if analise['pct_nulos'] <= 5 else ('🟡' if analise['pct_nulos'] <= 20 else '🔴')
            print(f"   {status} {col}: nulos={analise['nulos']:,} ({analise['pct_nulos']:.2f}%), únicos={analise['unicos']:,}")
            # Mostra top5 se disponível
            if analise['top5'] and not isinstance(analise['top5'], dict) or len(analise['top5']) > 0:
                print(f"      Top 5: {analise['top5']}")
    else:
        print("   ⚠️ Nenhuma coluna com 'PROGRAMA'.")

# =============================================
# 5. RELATÓRIO CONSOLIDADO POR COLUNA (agregando todos os arquivos)
# =============================================
print("\n" + "="*90)
print("📊 RELATÓRIO CONSOLIDADO POR COLUNA 'PROGRAMA' (agregado)")
print("="*90)

# Vamos consolidar: para cada coluna, qual o percentual de nulos médio e em quais arquivos aparece
consolidado = {}
for res in resultados:
    for col, analise in res['analise'].items():
        if col not in consolidado:
            consolidado[col] = {
                'arquivos': [],
                'pct_nulos_lista': [],
                'unicos_lista': [],
                'total_linhas_total': 0,
                'nulos_total': 0
            }
        consolidado[col]['arquivos'].append(res['arquivo'])
        consolidado[col]['pct_nulos_lista'].append(analise['pct_nulos'])
        consolidado[col]['unicos_lista'].append(analise['unicos'])
        consolidado[col]['total_linhas_total'] += analise['total']
        consolidado[col]['nulos_total'] += analise['nulos']

# Calcular percentual médio e classificação
for col, dados in consolidado.items():
    # Média simples dos percentuais (ou ponderada pelo total de linhas)
    pct_medio = dados['nulos_total'] / dados['total_linhas_total'] * 100 if dados['total_linhas_total'] > 0 else 0
    dados['pct_medio'] = round(pct_medio, 2)
    if pct_medio <= 5:
        dados['status'] = '🟢 Excelente'
    elif pct_medio <= 20:
        dados['status'] = '🟡 Razoável'
    else:
        dados['status'] = '🔴 Crítico'
    dados['arquivos_str'] = ', '.join(set(dados['arquivos']))

# Criar DataFrame para exibição
df_consolidado = pd.DataFrame([
    {
        'Coluna': col,
        'Arquivos': dados['arquivos_str'],
        'Pct_Nulos_Medio': dados['pct_medio'],
        'Status': dados['status'],
        'Unicos (min/max)': f"{min(dados['unicos_lista'])} / {max(dados['unicos_lista'])}"
    }
    for col, dados in consolidado.items()
])

df_consolidado = df_consolidado.sort_values('Pct_Nulos_Medio')
print(df_consolidado.to_string(index=False))

# =============================================
# 6. RECOMENDAÇÃO FINAL
# =============================================
print("\n" + "="*90)
print("✅ RECOMENDAÇÃO")
print("="*90)

excelentes = [col for col, dados in consolidado.items() if dados['status'] == '🟢 Excelente']
razoaveis = [col for col, dados in consolidado.items() if dados['status'] == '🟡 Razoável']
criticos = [col for col, dados in consolidado.items() if dados['status'] == '🔴 Crítico']

if excelentes:
    print(f"\n🟢 Colunas com ótima saúde (≤5% nulos): {', '.join(excelentes)}")
if razoaveis:
    print(f"\n🟡 Colunas com saúde razoável (5-20% nulos): {', '.join(razoaveis)} - Avalie se compensa usar (perda de até 20% das observações).")
if criticos:
    print(f"\n🔴 Colunas com muitos nulos (>20%): {', '.join(criticos)} - Evite usar ou faça imputação com cautela.")

print("\n🏁 Análise concluída!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Encontrei 13 arquivo(s):
   - SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2014.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2015.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2016.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2017.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2018.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2019.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2020.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2021.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2022.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2023.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2024.parquet
   - SICOR_OPERACAO_BASICA_ESTADO_2025.parquet

⏳ Analisando colunas com 'PROGRAMA' em cada arquivo...

   📂 SICOR_OPERACAO_BASICA_ESTADO_2013.parquet...


/tmp/ipykernel_2768/2409355165.py:49: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = pl.scan_parquet(caminho_parquet).schema


   📂 SICOR_OPERACAO_BASICA_ESTADO_2014.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2015.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2016.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2017.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2018.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2019.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2020.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2021.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2022.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2023.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2024.parquet...
   📂 SICOR_OPERACAO_BASICA_ESTADO_2025.parquet...

📋 COLUNAS COM 'PROGRAMA' ENCONTRADAS EM CADA ARQUIVO

📄 SICOR_OPERACAO_BASICA_ESTADO_2013.parquet (2,804,692 linhas):
   🟡 CD_PROGRAMA: nulos=410,344 (14.63%), únicos=21
      Top 5: {1.0: 1988564, 50.0: 213933, 999.0: 82787, 157.0: 61864, 70.0: 15763}
   🔴 CD_SUBPROGRAMA: nulos=2,804,626 (100.00%), únicos=7
      Top 5: {2.0: 44, 32.0: 10, 27.0: 7, 34.0: 2, 28.0: 2}

📄 SICOR_OPERACAO_BASICA_ESTA

# Transformando arquivo .CSV de dados complementares em .parquet.

In [ ]:
import polars as pl

# Cole aqui o caminho do seu arquivo Parquet no Colab
# Exemplo: '/content/SICOR_COMPLEMENTO_OPERACAO_BASICA_2019.parquet' ou o caminho do Google Drive
caminho_parquet = r'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet/SICOR_COMPLEMENTO_OPERACAO_BASICA.parquet'

try:
    # 1. Cria a referência Lazy para o arquivo
    lazy_df = pl.scan_parquet(caminho_parquet)

    # 2. Obtém o esquema (nomes das colunas e tipos de dados)
    esquema = lazy_df.schema

    print(f"📊 Total de colunas: {len(esquema)}\n")
    print("--- ESTRUTURA DE COLUNAS E TIPOS DE DADOS ---")
    for idx, (nome_coluna, tipo_dado) in enumerate(esquema.items(), start=1):
        print(f"{idx:02d}. {nome_coluna} -> {tipo_dado}")

    # 3. Lê apenas as primeiras 5 linhas para dar uma olhada na prévia dos dados
    print("\n--- PRÉVIA DOS DADOS (Primeiras 5 linhas) ---")
    df_preview = lazy_df.head(5).collect()

    # Configura para exibir todas as colunas no print sem cortar com '...'
    with pl.Config(tbl_cols=-1):
        print(df_preview)

except Exception as e:
    print(f"❌ Erro ao ler o arquivo Parquet: {e}")

/tmp/ipykernel_1048/4136139922.py:12: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  esquema = lazy_df.schema


📊 Total de colunas: 6

--- ESTRUTURA DE COLUNAS E TIPOS DE DADOS ---
01. #REF_BACEN -> Int64
02. NU_ORDEM -> Int64
03. REF_BACEN_EFETIVO -> Int64
04. AGENCIA_IF -> Int64
05. CD_IBGE_MUNICIPIO -> Int64
06. NUM_CEDULA_IF -> String

--- PRÉVIA DOS DADOS (Primeiras 5 linhas) ---
shape: (5, 6)
┌────────────┬──────────┬───────────────────┬────────────┬───────────────────┬───────────────────┐
│ #REF_BACEN ┆ NU_ORDEM ┆ REF_BACEN_EFETIVO ┆ AGENCIA_IF ┆ CD_IBGE_MUNICIPIO ┆ NUM_CEDULA_IF     │
│ ---        ┆ ---      ┆ ---               ┆ ---        ┆ ---               ┆ ---               │
│ i64        ┆ i64      ┆ i64               ┆ i64        ┆ i64               ┆ str               │
╞════════════╪══════════╪═══════════════════╪════════════╪═══════════════════╪═══════════════════╡
│ 3854564    ┆ 1        ┆ 20141159455       ┆ 239739     ┆ 1715150           ┆ 00000000004003190 │
│ 512703478  ┆ 1        ┆ 20211284392       ┆ 14007      ┆ 3171030           ┆ 1140C100140101001 │
│ 511209259  ┆ 1 

# Cruzando dados das planilhas do SICOR com a planilha auxiliar, para obter todos os códigos de todos os municípios, de acordo com o IBGE.

In [ ]:
import polars as pl
import glob
import os

# 1. Configuração dos caminhos no Colab / PC
pasta_sicor_origem = r'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet' # Pasta com os .parquet principais
pasta_sicor_destino = r'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Codigo_Municipios_IBGE'        # Pasta onde os arquivos finalizados serão salvos
caminho_complementar = r'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet/Planilha_Complementar_Parquet/SICOR_COMPLEMENTO_OPERACAO_BASICA.parquet'

# Cria a pasta de destino caso não exista
os.makedirs(pasta_sicor_destino, exist_ok=True)

# 2. Definição das chaves de cruzamento (AJUSTE OS NOMES CONFORME SEU CABEÇALHO)
# Exemplo de nomes comuns no SICOR: ['cd_ref_bacen', 'nr_ordem'] ou ['REF_BACEN', 'NUM_ORDEM']
chaves_cruzamento = ['#REF_BACEN', 'NU_ORDEM']

print("Preparando a tabela auxiliar complementar...")

try:
    # Prepara a tabela complementar em modo Lazy
    # (Se o seu arquivo complementar for .csv em vez de .parquet, use pl.scan_csv(caminho_complementar, separator=';', encoding='latin-1'))
    df_complemento = pl.scan_parquet(caminho_complementar)

    # Lista todos os arquivos Parquet da pasta
    arquivos_sicor = glob.glob(os.path.join(pasta_sicor_origem, "*.parquet"))
    print(f"📂 Encontrados {len(arquivos_sicor)} arquivos para enriquecer.\n")

    # 3. Processamento arquivo por arquivo em lote
    for idx, caminho_arquivo in enumerate(arquivos_sicor, start=1):
        nome_arquivo = os.path.basename(caminho_arquivo)
        caminho_saida = os.path.join(pasta_sicor_destino, f"enriquecido_{nome_arquivo}")

        print(f"[{idx}/{len(arquivos_sicor)}] Cruzando chaves e enriquecendo: {nome_arquivo}...")

        # O Polars faz o JOIN via streaming direto para o disco
        pl.scan_parquet(caminho_arquivo).join(
            df_complemento,
            on=chaves_cruzamento,
            how="left"  # Mantém todas as linhas do SICOR original e anexa as colunas da tabela complementar
        ).sink_parquet(
            caminho_saida,
            compression="snappy"
        )

        print(f"   ✅ Salvo com sucesso: {caminho_saida}")

    print("\n🎉 Todas as planilhas da pasta foram enriquecidas com sucesso!")

except Exception as e:
    print(f"❌ Erro durante o processamento: {e}")

Preparando a tabela auxiliar complementar...
📂 Encontrados 13 arquivos para enriquecer.

[1/13] Cruzando chaves e enriquecendo: SICOR_OPERACAO_BASICA_ESTADO_2013.parquet...
   ✅ Salvo com sucesso: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Codigo_Municipios_IBGE/enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
[2/13] Cruzando chaves e enriquecendo: SICOR_OPERACAO_BASICA_ESTADO_2014.parquet...
   ✅ Salvo com sucesso: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Codigo_Municipios_IBGE/enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2014.parquet
[3/13] Cruzando chaves e enriquecendo: SICOR_OPERACAO_BASICA_ESTADO_2015.parquet...
   ✅ Salvo com sucesso: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Codigo_Municipios_IBGE/enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2015.parquet
[4/13] Cruzando chaves e enriquecendo: SICOR_OP

# Cruzando as planilhas enriquecidas do SICOR com a planilha auxiliar com os nomes dos municípios e seus códigos, para, assim, obter os nomes dos municípios.

In [ ]:
import polars as pl
import glob
import os

# 1. Configuração dos caminhos no Google Drive / Colab
pasta_sicor_origem = r'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Codigo_Municipios_IBGE'
pasta_sicor_destino = r'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE'
caminho_auxiliar_ibge = r'/content/drive/MyDrive/TCC - Ideias/Planilhas para Relacionar - SICOR/MunicipiosSicor.csv'

# Cria a pasta de destino caso não exista
os.makedirs(pasta_sicor_destino, exist_ok=True)

# 2. Definição das colunas chaves do município (AJUSTE SEUS NOMES CONFORME OS CABEÇALHOS)
coluna_ibge_sicor = 'CD_IBGE_MUNICIPIO'  # Nome da coluna que veio da planilha complementar do SICOR
coluna_ibge_auxiliar = 'COD_MUNICIPIO_IBGE'    # Nome da coluna de código do município na tabela auxiliar IBGE

print("Preparando a tabela auxiliar de municípios do IBGE...")

try:
    # 3. Prepara a tabela auxiliar do IBGE em modo Lazy
    # Se a tabela auxiliar for CSV:
    import polars as pl

# 1. Read eagerly using latin-1 encoding
# 2. Convert to lazy dataframe using .lazy()
    df_ibge = (
        pl.read_csv(
            caminho_auxiliar_ibge,
            separator=";",
            encoding="latin-1",  # read_csv supports latin-1 natively
            infer_schema_length=10000,
        )
        .lazy()
        .with_columns(pl.col(coluna_ibge_auxiliar).cast(pl.String))
    )

    # Nota: Se a sua tabela auxiliar for .parquet, troque 'pl.scan_csv' acima por:
    # df_ibge = pl.scan_parquet(caminho_auxiliar_ibge).with_columns(
    #     pl.col(coluna_ibge_auxiliar).cast(pl.String)
    # )

    # Lista todos os arquivos Parquet da pasta da etapa 1
    arquivos_sicor = glob.glob(os.path.join(pasta_sicor_origem, "*.parquet"))
    print(f"📂 Encontrados {len(arquivos_sicor)} arquivos para anexar as cidades.\n")

    # 4. Processamento arquivo por arquivo em lote via streaming
    for idx, caminho_arquivo in enumerate(arquivos_sicor, start=1):
        nome_arquivo = os.path.basename(caminho_arquivo)
        caminho_saida = os.path.join(pasta_sicor_destino, f"final_{nome_arquivo}")

        print(f"[{idx}/{len(arquivos_sicor)}] Cruzando municípios e gerando: {nome_arquivo}...")

        # O Polars faz o JOIN via streaming direto para o disco
        pl.scan_parquet(caminho_arquivo).with_columns(
            pl.col(coluna_ibge_sicor).cast(pl.String)
        ).join(
            df_ibge,
            left_on=coluna_ibge_sicor,
            right_on=coluna_ibge_auxiliar,
            how="left"  # Preserva todas as linhas da base do SICOR
        ).sink_parquet(
            caminho_saida,
            compression="snappy"
        )

        print(f"   ✅ Salvo com sucesso: {caminho_saida}")

    print("\n🎉 Todas as planilhas foram enriquecidas com os nomes das cidades!")

except Exception as e:
    print(f"❌ Erro durante o processamento: {e}")

Preparando a tabela auxiliar de municípios do IBGE...
📂 Encontrados 13 arquivos para anexar as cidades.

[1/13] Cruzando municípios e gerando: enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet...
   ✅ Salvo com sucesso: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
[2/13] Cruzando municípios e gerando: enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2014.parquet...
   ✅ Salvo com sucesso: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2014.parquet
[3/13] Cruzando municípios e gerando: enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2015.parquet...
   ✅ Salvo com sucesso: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2015.

# Explorando as planilhas com os nomes das cidades

In [ ]:
# =============================================
# INSTALAÇÃO E IMPORTS
# =============================================
# !pip install polars plotly pyarrow -U

import polars as pl
import plotly.express as px
import glob
import os
import re
from typing import Optional, Dict, List, Any

# =============================================
# CONFIGURAÇÕES GLOBAIS
# =============================================
CAMINHO_PASTA = '/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE'
COLUNA_VALOR = 'VL_PARC_CREDITO'
COLUNA_JUROS = 'VL_JUROS'
COLUNA_ESTADO = 'CD_ESTADO'
COLUNA_PROGRAMA = 'CD_PROGRAMA'

# Configuração para inspeção de todas as colunas
AMOSTRA_INSPERCAO = 1000  # Linhas por arquivo para preview completo

# =============================================
# FUNÇÕES AUXILIARES ROBUSTAS
# =============================================
def extrair_ano_do_nome(nome_arquivo: str) -> Optional[int]:
    match = re.search(r'(\d{4})', nome_arquivo)
    return int(match.group(1)) if match else None

def tratar_valor_monetario(coluna: str, dividir_por_100: bool = True) -> pl.Expr:
    expr = (
        pl.col(coluna)
        .cast(pl.String, strict=False)
        .str.replace_all(r'\.', '')
        .str.replace(',', '.')
        .cast(pl.Float64, strict=False)
    )
    if dividir_por_100:
        expr = expr / 100.0
    return expr.alias(coluna)

def inspecionar_todas_colunas(caminho: str, n_linhas: int = AMOSTRA_INSPERCAO) -> pl.DataFrame:
    """
    Carrega uma amostra com TODAS as colunas do parquet para exploração.
    Útil para validar schema, tipos e valores brutos antes das agregações.
    """
    try:
        df = pl.read_parquet(caminho, n_rows=n_linhas)
        print(f"\n🔍 INSPEÇÃO: {os.path.basename(caminho)}")
        print(f"   Shape: {df.shape}")
        print(f"   Colunas ({len(df.columns)}): {df.columns}")
        print(f"   Schema:\n{df.schema}")
        print(f"\n   Primeiras 5 linhas:")
        print(df.head(5))
        return df
    except Exception as e:
        print(f"❌ Erro na inspeção de {os.path.basename(caminho)}: {e}")
        return pl.DataFrame()

def processar_arquivo_parquet(caminho: str, ano: int) -> Optional[Dict[str, Any]]:
    try:
        lf = pl.scan_parquet(caminho)
        schema = lf.collect_schema()
        colunas_disponiveis = set(schema.keys())

        if COLUNA_VALOR not in colunas_disponiveis:
            print(f"⚠️  {os.path.basename(caminho)}: '{COLUNA_VALOR}' não encontrada. Pulando.")
            return None

        lf = lf.with_columns(tratar_valor_monetario(COLUNA_VALOR))
        tem_juros = COLUNA_JUROS in colunas_disponiveis
        if tem_juros:
            lf = lf.with_columns(tratar_valor_monetario(COLUNA_JUROS))

        resultados = {}

        # Agregações globais
        agg_exprs = [pl.sum(COLUNA_VALOR).alias('total_credito'), pl.len().alias('total_operacoes')]
        if tem_juros:
            agg_exprs.append(pl.mean(COLUNA_JUROS).alias('juros_medio'))
        resultados['global'] = lf.select(agg_exprs).collect().to_dicts()[0]

        # Por Estado
        if COLUNA_ESTADO in colunas_disponiveis:
            estado_exprs = [pl.sum(COLUNA_VALOR).alias('TOTAL_CREDITO'), pl.len().alias('OPERACOES')]
            if tem_juros:
                estado_exprs.append(pl.mean(COLUNA_JUROS).alias('JUROS_MEDIO'))
            resultados['estado'] = (
                lf.group_by(COLUNA_ESTADO).agg(estado_exprs)
                .collect().with_columns(pl.lit(ano).alias('ANO'))
            )

        # Top 10 Programas
        if COLUNA_PROGRAMA in colunas_disponiveis:
            resultados['programa'] = (
                lf.group_by(COLUNA_PROGRAMA)
                .agg([pl.sum(COLUNA_VALOR).alias('TOTAL_CREDITO'), pl.len().alias('OPERACOES')])
                .sort('TOTAL_CREDITO', descending=True).head(10)
                .collect().with_columns(pl.lit(ano).alias('ANO'))
            )

        return {'ano': ano, **resultados}

    except Exception as e:
        print(f"❌ Erro ao processar {os.path.basename(caminho)}: {e}")
        return None

# =============================================
# PIPELINE PRINCIPAL COM INSPEÇÃO COMPLETA
# =============================================
# =============================================
# PIPELINE PRINCIPAL COM INSPEÇÃO COMPLETA (CORRIGIDO)
# =============================================
def explorar_dados_sicor(inspecionar_primeiro: bool = True):
    arquivos = sorted(glob.glob(os.path.join(CAMINHO_PASTA, '*.parquet')))
    print(f"📂 Encontrados {len(arquivos)} arquivos parquet.")

    amostras_completas = []
    df_todas_colunas = pl.DataFrame()  # Inicializa vazio para garantir retorno

    if inspecionar_primeiro:
        print("\n" + "="*60)
        print("🔎 MODO INSPEÇÃO: CARREGANDO TODAS AS COLUNAS (AMOSTRA)")
        print("="*60)

        for arq in arquivos[:3]:
            amostra = inspecionar_todas_colunas(arq)
            if not amostra.is_empty():
                ano = extrair_ano_do_nome(os.path.basename(arq))
                amostras_completas.append({'ano': ano, 'df': amostra})

        if amostras_completas:
            # ✅ CORREÇÃO: vertical_relaxed resolve incompatibilidade de tipos entre anos
            dfs_para_concat = [
                a['df'].with_columns(pl.lit(a['ano']).alias('ANO_REF'))
                for a in amostras_completas
            ]

            try:
                df_todas_colunas = pl.concat(dfs_para_concat, how="vertical_relaxed")
                print(f"\n✅ DataFrame consolidado com TODAS as colunas disponível como 'df_todas_colunas'")
                print(f"   Shape total: {df_todas_colunas.shape}")
                print(f"   Colunas ({len(df_todas_colunas.columns)}): {df_todas_colunas.columns}")
            except Exception as e:
                print(f"⚠️  Falha no concat relaxed: {e}")
                print("   Tentando concat diagonal (preserva todas as colunas mesmo se ausentes)...")
                df_todas_colunas = pl.concat(dfs_para_concat, how="diagonal_relaxed")
                print(f"   ✅ Sucesso com diagonal_relaxed. Shape: {df_todas_colunas.shape}")
        else:
            print("⚠️  Nenhuma amostra válida carregada na inspeção.")

    # --- ETAPA DE AGREGAÇÃO (sem alterações) ---
    print("\n" + "="*60)
    print("📊 PROCESSANDO AGREGADOS PARA VISUALIZAÇÕES")
    print("="*60)

    agregados = []
    for arq in arquivos:
        ano = extrair_ano_do_nome(os.path.basename(arq))
        if ano is None:
            continue
        print(f"⚙️  Processando {os.path.basename(arq)} ({ano})...")
        resultado = processar_arquivo_parquet(arq, ano)
        if resultado:
            agregados.append(resultado)

    if not agregados:
        print("❌ Nenhum dado válido processado.")
        return df_todas_colunas

    # =============================================
    # VISUALIZAÇÕES (código idêntico ao anterior)
    # =============================================
    df_evolucao = pl.DataFrame([
        {'ANO': a['ano'],
         'TOTAL_CREDITO_BI': a['global']['total_credito'] / 1e9,
         'OPERACOES': a['global']['total_operacoes']}
        for a in agregados
    ]).sort('ANO').to_pandas()

    fig1 = px.bar(df_evolucao, x='ANO', y='TOTAL_CREDITO_BI',
                  title='💰 Total de Crédito por Ano (Bilhões de R$)',
                  color='ANO', color_continuous_scale='Viridis')
    fig1.update_traces(texttemplate='%{y:.1f} Bi', textposition='outside')
    fig1.update_layout(yaxis_title="Bilhões de R$", showlegend=False)
    fig1.show()

    ano_recente = max(a['ano'] for a in agregados)
    dados_estado_recente = next((a['estado'] for a in agregados if a['ano'] == ano_recente), None)

    if dados_estado_recente is not None:
        top_estados = (dados_estado_recente
                       .sort('TOTAL_CREDITO', descending=True).head(10)
                       .with_columns((pl.col('TOTAL_CREDITO') / 1e9).alias('TOTAL_CREDITO_BI'))
                       .to_pandas())
        fig2 = px.bar(top_estados, x=COLUNA_ESTADO, y='TOTAL_CREDITO_BI',
                      title=f'🏆 Top 10 Estados - {ano_recente} (Bilhões de R$)',
                      color='TOTAL_CREDITO_BI', color_continuous_scale='Blues')
        fig2.update_traces(texttemplate='%{y:.1f} Bi', textposition='outside')
        fig2.update_layout(yaxis_title="Bilhões de R$")
        fig2.show()

    dados_programa_recente = next((a['programa'] for a in agregados if a['ano'] == ano_recente), None)
    if dados_programa_recente is not None:
        fig3 = px.pie(dados_programa_recente.to_pandas(),
                      values='OPERACOES', names=COLUNA_PROGRAMA,
                      title=f'📊 Top 10 Programas - {ano_recente}', hole=0.3)
        fig3.show()

    dfs_juros = [
        a['estado'].select([COLUNA_ESTADO, 'JUROS_MEDIO', 'ANO'])
        for a in agregados
        if 'estado' in a and 'JUROS_MEDIO' in a['estado'].columns
    ]
    if len(dfs_juros) >= 2 and dados_estado_recente is not None:
        top10_lista = (dados_estado_recente.sort('OPERACOES', descending=True)
                       .head(10)[COLUNA_ESTADO].to_list())
        df_juros = (pl.concat(dfs_juros)
                    .filter(pl.col(COLUNA_ESTADO).is_in(top10_lista))
                    .to_pandas())
        fig4 = px.bar(df_juros, x=COLUNA_ESTADO, y='JUROS_MEDIO', color='ANO',
                      barmode='group',
                      title='📉 Juros Médios - Top 10 Estados (Comparativo Anual)',
                      labels={'JUROS_MEDIO': 'Taxa Média (%)'})
        fig4.show()

    resumo = df_evolucao.copy()
    resumo['TOTAL_CREDITO_FMT'] = resumo['TOTAL_CREDITO_BI'].apply(lambda x: f'R$ {x:,.2f} Bi')
    resumo['OPERACOES_FMT'] = resumo['OPERACOES'].apply(lambda x: f'{x:,.0f}')
    print("\n📋 RESUMO EXECUTIVO:")
    print(resumo[['ANO', 'TOTAL_CREDITO_FMT', 'OPERACOES_FMT']].to_string(index=False))

    return df_todas_colunas

# =============================================
# EXECUÇÃO
# =============================================
if __name__ == "__main__":
    # Retorna DataFrame com todas as colunas para exploração adicional
    df_completo = explorar_dados_sicor(inspecionar_primeiro=True)

    # Exemplo de uso pós-execução:
    # df_completo.filter(pl.col('ANO_REF') == 2023).select(['CD_ESTADO', 'VL_PARC_CREDITO', ...])

📂 Encontrados 13 arquivos parquet.

🔎 MODO INSPEÇÃO: CARREGANDO TODAS AS COLUNAS (AMOSTRA)

🔍 INSPEÇÃO: final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
   Shape: (1000, 56)
   Colunas (56): ['#REF_BACEN', 'NU_ORDEM', 'CNPJ_IF', 'DT_EMISSAO', 'DT_VENCIMENTO', 'CD_INST_CREDITO', 'CD_CATEG_EMITENTE', 'CD_FONTE_RECURSO', 'CNPJ_AGENTE_INVEST', 'CD_ESTADO', 'CD_REF_BACEN_INVESTIMENTO', 'CD_TIPO_SEGURO', 'CD_EMPREENDIMENTO', 'CD_PROGRAMA', 'CD_TIPO_ENCARG_FINANC', 'CD_TIPO_IRRIGACAO', 'CD_TIPO_AGRICULTURA', 'CD_FASE_CICLO_PRODUCAO', 'CD_TIPO_CULTIVO', 'CD_TIPO_INTGR_CONSOR', 'CD_TIPO_GRAO_SEMENTE', 'VL_ALIQ_PROAGRO', 'VL_JUROS', 'VL_PRESTACAO_INVESTIMENTO', 'VL_PREV_PROD', 'VL_QUANTIDADE', 'VL_RECEITA_BRUTA_ESPERADA', 'VL_PARC_CREDITO', 'VL_REC_PROPRIO', 'VL_PERC_RISCO_STN', 'VL_PERC_RISCO_FUNDO_CONST', 'VL_REC_PROPRIO_SRV', 'VL_AREA_FINANC', 'CD_SUBPROGRAMA', 'VL_PRODUTIV_OBTIDA', 'DT_FIM_COLHEITA', 'DT_FIM_PLANTIO', 'DT_INIC_COLHEITA', 'DT_INIC_PLANTIO', 'VL_JUROS_ENC_FINAN_POSF


📋 RESUMO EXECUTIVO:
 ANO TOTAL_CREDITO_FMT OPERACOES_FMT
2013      R$ 139.39 Bi     2,804,692
2014      R$ 164.43 Bi     2,584,654
2015      R$ 154.15 Bi     2,330,156
2016      R$ 117.62 Bi     1,576,888
2017      R$ 167.02 Bi     2,109,447
2018      R$ 180.88 Bi     2,033,300
2019      R$ 178.95 Bi     1,860,900
2020      R$ 207.00 Bi     1,924,778
2021      R$ 294.99 Bi     1,989,446
2022      R$ 361.98 Bi     1,990,945
2023      R$ 406.40 Bi     2,129,134
2024      R$ 379.78 Bi     2,328,948
2025      R$ 358.14 Bi     2,442,939


# Cruzando as planilhas enriquecidas do SICOR com os nomes dos municípios com as planilhas auxiliares, para obter dados importantes

## Como: Fonte dos Recursos, Modalidade do Programa, o Programa em si, o Produto e o que foi cultivado.

In [ ]:
# ==============================================================================
# VERSÃO OTIMIZADA PARA COLAB FREE - MENOR USO DE RAM
# ==============================================================================
# Se necessário, atualize antes:
# !pip install -U polars pyarrow

import os
import glob
import gc

# ------------------------------------------------------------------------------
# Variáveis de ambiente para reduzir memória/fragmentação e limitar threads.
# Precisam ser definidas ANTES de importar o Polars.
# Se o Polars já foi importado nesta sessão, reinicie o runtime.
# ------------------------------------------------------------------------------
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["POLARS_MAX_THREADS"] = "2"   # Colab Free normalmente tem 2 vCPUs
os.environ["POLARS_VERBOSE"] = "0"

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

# ==============================================================================
# CONFIGURAÇÕES DE CAMINHOS E ESCOPO
# ==============================================================================
PASTA_SICOR_ORIGEM = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"
    "Parquet_Final_Cidades_IBGE"
)

PASTA_AUXILIAR = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Planilhas para Relacionar - SICOR"
)

PASTA_DESTINO = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"
    "Parquet_Enriquecido_Completo"
)

ANOS_ESCOPO = {str(ano) for ano in range(2019, 2026)}

# ------------------------------------------------------------------------------
# PARÂMETROS DE MEMÓRIA
# ------------------------------------------------------------------------------
# Se ainda estourar RAM, reduza para 10_000 ou 5_000.
# Se estiver rápido e com RAM folgada, pode aumentar para 50_000 ou 100_000.
TAMANHO_BLOCO = 25_000

# "snappy" é mais rápido e costuma ser mais leve para CPU no Colab Free.
# Se quiser arquivos menores, use "zstd", mas pode ficar mais lento.
COMPRESSION = "snappy"

os.makedirs(PASTA_DESTINO, exist_ok=True)

MAPA_AUXILIARES = {
    "FonteRecursos.csv": "CD_FONTE_RECURSO",
    "Subprogramas.csv": "CD_SUBPROGRAMA",
    "ProgramaModalidade.csv": "CD_PROGRAMA",
    "Programa.csv": "CD_PROGRAMA",
    "Empreendimento.csv": "CD_EMPREENDIMENTO",
    "InstrumentoCredito.csv": "CD_INST_CREDITO",
    "TipoGarantiaEmpreendimento.csv": "CD_TIPO_SEGURO",
}

# ==============================================================================
# FUNÇÕES AUXILIARES
# ==============================================================================

def normalizar_chave_expr(nome_coluna: str) -> pl.Expr:
    """
    Normaliza uma coluna de código para Int64.

    Estratégia:
    - Converte para String
    - Extrai o primeiro sequência de dígitos
    - Converte para Int64

    Isso evita problemas com códigos como:
    - "00123"
    - "CODIGO 123"
    - 123.0
    - "123/45"
    """
    return (
        pl.col(nome_coluna)
        .cast(pl.String)
        .str.extract(r"(\d+)")
        .cast(pl.Int64, strict=False)
    )


def tipo_efetivo(tipo_arrow):
    """
    Se a coluna for dictionary/categorical no Arrow, pega o tipo real dos valores.
    """
    while pa.types.is_dictionary(tipo_arrow):
        tipo_arrow = tipo_arrow.value_type
    return tipo_arrow


def expr_chave_base(nome_coluna: str, tipo_arrow) -> pl.Expr:
    """
    Expressão para normalizar a chave no arquivo principal.

    Se a coluna já for numérica, só faz cast para Int64.
    Se for texto/outro, extrai dígitos.
    """
    if pa.types.is_floating(tipo_arrow):
        return (
            pl.col(nome_coluna)
            .fill_nan(None)
            .cast(pl.Int64, strict=False)
            .alias(nome_coluna)
        )

    if pa.types.is_integer(tipo_arrow):
        return (
            pl.col(nome_coluna)
            .cast(pl.Int64, strict=False)
            .alias(nome_coluna)
        )

    return (
        pl.col(nome_coluna)
        .cast(pl.String)
        .str.extract(r"(\d+)")
        .cast(pl.Int64, strict=False)
        .alias(nome_coluna)
    )


def encontrar_coluna_origem(cols: list[str], chave_alvo: str) -> str:
    """
    Escolhe a coluna de origem da chave na tabela auxiliar.
    Prioriza:
    - #CODIGO
    - CODIGO
    - nome da própria chave alvo
    Caso contrário, usa a primeira coluna.
    """
    alvo = chave_alvo.upper()

    for c in cols:
        u = c.upper()
        if u in ("#CODIGO", "CODIGO", alvo):
            return c

    return cols[0]


def carregar_auxiliar(nome_arquivo: str, nova_chave: str) -> pl.DataFrame | None:
    """
    Carrega a tabela auxiliar, normaliza a chave, remove nulos e deduplica.

    IMPORTANTE:
    A deduplicação por chave é essencial para evitar que o LEFT JOIN
    multiplique as linhas da base principal.
    """
    caminho = os.path.join(PASTA_AUXILIAR, nome_arquivo)

    if not os.path.exists(caminho):
        print(f"⚠️ Arquivo auxiliar não encontrado: {nome_arquivo}")
        return None

    # Alguns arquivos podem estar em latin-1.
    # Se o Polars recusar o encoding, tenta utf8-lossy.
    try:
        df = pl.read_csv(
            caminho,
            separator=";",
            encoding="latin-1",
            infer_schema_length=0,
            ignore_errors=True,
        )
    except Exception:
        df = pl.read_csv(
            caminho,
            separator=";",
            encoding="utf8-lossy",
            infer_schema_length=0,
            ignore_errors=True,
        )

    if not df.columns or df.is_empty():
        print(f"⚠️ Tabela auxiliar vazia: {nome_arquivo}")
        return None

    # Remove BOM e espaços nas bordas dos nomes de colunas.
    df = df.rename(
        {c: c.replace("\ufeff", "").strip() for c in df.columns}
    )

    col_origem = encontrar_coluna_origem(df.columns, nova_chave)

    # Nome temporário para evitar conflito caso a coluna origem tenha o mesmo
    # nome da chave final.
    temp_chave = f"__{nova_chave}_aux__"

    # Colunas que devem ser removidas:
    # - coluna origem original
    # - eventual coluna já existente com o nome da chave final
    colunas_para_remover = [
        c for c in {col_origem, nova_chave} if c in df.columns
    ]

    lf = df.lazy().with_columns(
        normalizar_chave_expr(col_origem).alias(temp_chave)
    )

    if colunas_para_remover:
        lf = lf.drop(colunas_para_remover)

    df_aux = (
        lf
        .rename({temp_chave: nova_chave})
        .filter(pl.col(nova_chave).is_not_null())
        .unique(
            subset=[nova_chave],
            keep="first",
            maintain_order=False,
        )
        .collect()
    )

    if df_aux.is_empty():
        print(f"⚠️ Tabela auxiliar sem chaves válidas: {nome_arquivo}")
        return None

    return df_aux


def enriquecer(
    df: pl.DataFrame,
    exprs_chave: list[pl.Expr],
    joins_info: list[tuple[str, pl.DataFrame, str]],
) -> pl.DataFrame:
    """
    Aplica normalização das chaves e os joins auxiliares em um bloco.
    """
    lf = df.lazy()

    if exprs_chave:
        lf = lf.with_columns(exprs_chave)

    for chave, df_aux, sufixo in joins_info:
        lf = lf.join(
            df_aux.lazy(),
            on=chave,
            how="left",
            suffix=sufixo,
        )

    return lf.collect()


def processar_arquivo(
    pf: pq.ParquetFile,
    caminho_saida: str,
    exprs_chave: list[pl.Expr],
    joins_info: list[tuple[str, pl.DataFrame, str]],
) -> int:
    """
    Processa um arquivo Parquet em blocos e escreve a saída em blocos.
    Isso evita materializar o arquivo inteiro na RAM.
    """
    writer = None
    total_linhas = 0

    try:
        for batch in pf.iter_batches(batch_size=TAMANHO_BLOCO):
            df = pl.from_arrow(batch)
            df = enriquecer(df, exprs_chave, joins_info)

            tabela = df.to_arrow()

            if writer is None:
                writer = pq.ParquetWriter(
                    caminho_saida,
                    tabela.schema,
                    compression=COMPRESSION,
                )

            writer.write_table(
                tabela,
                row_group_size=TAMANHO_BLOCO,
            )

            total_linhas += df.height

            # Limpeza agressiva para ambiente com RAM limitada.
            del df
            del tabela
            del batch
            gc.collect()

        # Caso o arquivo de origem esteja vazio.
        if writer is None:
            df_vazio = pl.from_arrow(pf.schema_arrow.empty_table())
            df_vazio = enriquecer(df_vazio, exprs_chave, joins_info)

            tabela_vazia = df_vazio.to_arrow()

            with pq.ParquetWriter(
                caminho_saida,
                tabela_vazia.schema,
                compression=COMPRESSION,
            ) as writer_vazio:
                writer_vazio.write_table(tabela_vazia)

            total_linhas = 0
        else:
            writer.close()

        return total_linhas

    except Exception:
        # Se falhar, fecha e remove arquivo parcial para evitar lixo.
        if writer is not None:
            writer.close()

        if os.path.exists(caminho_saida):
            try:
                os.remove(caminho_saida)
            except Exception:
                pass

        raise


# ==============================================================================
# EXECUÇÃO PRINCIPAL
# ==============================================================================

arquivos = [
    a
    for a in sorted(glob.glob(os.path.join(PASTA_SICOR_ORIGEM, "*.parquet")))
    if any(
        os.path.basename(a).endswith(f"_{ano}.parquet")
        for ano in ANOS_ESCOPO
    )
]

if not arquivos:
    print(f"⚠️ Nenhum arquivo encontrado em: {PASTA_SICOR_ORIGEM}")

else:
    print(f"🔎 Lendo schemas de {len(arquivos)} arquivos...")

    # ------------------------------------------------------------------------------
    # 1) Descobrir quais chaves realmente existem em cada arquivo.
    #    Isso evita carregar/applyar joins desnecessários.
    # ------------------------------------------------------------------------------
    meta_arquivos = {}
    todas_chaves = set()

    for arq in arquivos:
        pf_meta = pq.ParquetFile(arq)
        schema = pf_meta.schema_arrow

        nomes_colunas = set(schema.names)

        chaves_presentes = sorted(
            {
                chave
                for chave in MAPA_AUXILIARES.values()
                if chave in nomes_colunas
            }
        )

        meta_arquivos[arq] = (schema, chaves_presentes)
        todas_chaves.update(chaves_presentes)

        del pf_meta

    print(f"🚀 Iniciando processamento otimizado de {len(arquivos)} arquivos...")
    print(f"📦 Tamanho do bloco: {TAMANHO_BLOCO:,} linhas")

    # ------------------------------------------------------------------------------
    # 2) Carregar tabelas auxiliares apenas se a chave for usada.
    #    As tabelas auxiliares são deduplicadas pela chave.
    # ------------------------------------------------------------------------------
    print("⏳ Carregando tabelas auxiliares necessárias...")

    dfs_aux = {}

    for nome_csv, chave in MAPA_AUXILIARES.items():
        if chave not in todas_chaves:
            continue

        df_aux = carregar_auxiliar(nome_csv, chave)

        if df_aux is not None:
            dfs_aux[nome_csv] = df_aux
            print(f"📄 {nome_csv}: {df_aux.height:,} linhas únicas")

            if df_aux.height > 500_000:
                print(
                    f"⚠️ {nome_csv} está grande. "
                    "Se possível, selecione menos colunas para reduzir RAM."
                )

    if not dfs_aux:
        print("⚠️ Nenhuma tabela auxiliar foi carregada.")

    # ------------------------------------------------------------------------------
    # 3) Processar arquivo por arquivo em blocos.
    # ------------------------------------------------------------------------------
    for idx, arq in enumerate(arquivos, 1):
        nome_base = os.path.basename(arq)
        caminho_saida = os.path.join(PASTA_DESTINO, f"full_{nome_base}")

        schema, chaves_presentes = meta_arquivos[arq]

        # Expressões para normalizar as chaves no arquivo principal.
        exprs_chave = []

        for coluna in chaves_presentes:
            tipo_arrow = tipo_efetivo(schema.field(coluna).type)
            exprs_chave.append(expr_chave_base(coluna, tipo_arrow))

        # Joins que realmente serão usados neste arquivo.
        joins_info = []

        for nome_csv, df_aux in dfs_aux.items():
            chave = MAPA_AUXILIARES[nome_csv]

            if chave in chaves_presentes:
                sufixo = f"_{os.path.splitext(nome_csv)[0]}"
                joins_info.append((chave, df_aux, sufixo))

        pf = pq.ParquetFile(arq)

        try:
            total_linhas = processar_arquivo(
                pf=pf,
                caminho_saida=caminho_saida,
                exprs_chave=exprs_chave,
                joins_info=joins_info,
            )

            print(
                f"[{idx}/{len(arquivos)}] ✅ {nome_base} | "
                f"{total_linhas:,} linhas"
            )

        except Exception as e:
            print(
                f"[{idx}/{len(arquivos)}] ❌ {nome_base} | Erro: {e}"
            )

        finally:
            del pf
            gc.collect()

    print("\n🎉 Processamento finalizado!")

🔎 Lendo schemas de 7 arquivos...
🚀 Iniciando processamento otimizado de 7 arquivos...
📦 Tamanho do bloco: 25,000 linhas
⏳ Carregando tabelas auxiliares necessárias...
📄 FonteRecursos.csv: 37 linhas únicas
📄 Subprogramas.csv: 85 linhas únicas
📄 ProgramaModalidade.csv: 39 linhas únicas
📄 Programa.csv: 40 linhas únicas
📄 Empreendimento.csv: 3,284 linhas únicas
📄 InstrumentoCredito.csv: 11 linhas únicas
📄 TipoGarantiaEmpreendimento.csv: 5 linhas únicas
[1/7] ✅ final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2019.parquet | 1,860,900 linhas
[2/7] ✅ final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2020.parquet | 1,924,778 linhas
[3/7] ✅ final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2021.parquet | 1,989,446 linhas
[4/7] ✅ final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2022.parquet | 1,990,945 linhas
[5/7] ✅ final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2023.parquet | 2,129,134 linhas
[6/7] ✅ final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2024.parquet | 2,328,948 linhas
[7/7] ✅ final_enriquecido

### 🔍 Verificação Rápida de Caminhos (CLI)
Use os comandos abaixo para listar os arquivos e pastas e confirmar os nomes exatos no Drive.

In [ ]:
# Listar as subpastas para confirmar o nome da pasta Cidades_IBGE (case-sensitive)
!ls -d "/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"*/

# Listar arquivos Parquet para validar o padrão do nome (_YYYY)
!find "/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/" -name "*.parquet" | head -n 10

'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet/'
'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Codigo_Municipios_IBGE/'
'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Enriquecido_Completo/'
'/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/'
/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2014.parquet
/content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Final_Cidades_IBGE/final_enriquecido_SICOR_OPE

# Análise Completa: Todas as Colunas dos Arquivos do SICOR com Cidades

In [ ]:
# ==============================================================================
# AUDITORIA DE SAÚDE DAS COLUNAS - OTIMIZADO PARA COLOMB FREE (SEM PANDAS)
# ==============================================================================
# Baseado na arquitetura de processamento em blocos (chunking) com PyArrow + Polars.
# Evita estourar a RAM ao calcular nulos e únicos de arquivos gigantescos.

import os
import gc
import glob
import pyarrow.parquet as pq
import polars as pl

# ------------------------------------------------------------------------------
# Variáveis de ambiente para reduzir memória/fragmentação (MESMAS DO SEU CÓDIGO)
# ------------------------------------------------------------------------------
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["POLARS_MAX_THREADS"] = "2"
os.environ["POLARS_VERBOSE"] = "0"

# ==============================================================================
# CONFIGURAÇÕES
# ==============================================================================
PASTA_PARQUETS = (
     "/content/drive/MyDrive/TCC - Ideias/"
    "Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"
    "Parquet_Enriquecido_Completo"
)

PASTA_RELATORIOS = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Relatorios_Saude_Colunas"
)
os.makedirs(PASTA_RELATORIOS, exist_ok=True)

# Tamanho do bloco para não estourar RAM (25k a 50k é o ideal no Colab Free)
TAMANHO_BLOCO = 25_000

# Limite de memória para o set de únicos. Se passar disso, paramos de contar
# e marcamos como "Alta Cardinalidade" para evitar OOM (Out of Memory).
MAX_UNICOS_TRACK = 500_000

# ------------------------------------------------------------------------------
# COLUNAS PROIBIDAS / VAZAMENTO (Definidas na Justificativa do TCC)
# ------------------------------------------------------------------------------
COLUNAS_IDENTIFICADORES = {
    '#REF_BACEN', 'NU_ORDEM', 'CNPJ_IF', 'CD_INST_CREDITO',
    'CNPJ_AGENTE_INVEST', 'CD_EMPREENDIMENTO', 'CD_CONTRATO_STN',
    'CD_CNPJ_CADASTRANTE', 'NUM_CEDULA_IF', 'AGENCIA_IF', 'REF_BACEN_EFETIVO'
}

COLUNAS_REDUNDANTES_GEO = {
    'NOME_MUNICIPIO', 'COD_UF', 'NOME_UF', 'COD_UF_IBGE',
    '#COD_MUNICIPIO_CADMU', 'VL_AREA_INFORMADA'
}

COLUNAS_VAZAMENTO_DADOS = {
    'VL_PRODUTIV_OBTIDA'  # ⚠️ DATA LEAKAGE: Só se conhece no fim da safra!
}

COLUNAS_PROIBIDAS = COLUNAS_IDENTIFICADORES | COLUNAS_REDUNDANTES_GEO | COLUNAS_VAZAMENTO_DADOS

# ==============================================================================
# FUNÇÃO DE ANÁLISE EM BLOCOS
# ==============================================================================

def analisar_saude_arquivo(caminho_arquivo: str) -> pl.DataFrame:
    """
    Analisa a saúde das colunas de um Parquet gigante processando em blocos.
    """
    nome_base = os.path.basename(caminho_arquivo)
    print(f"\n🔍 Analisando: {nome_base}")

    pf = pq.ParquetFile(caminho_arquivo)
    schema = pf.schema_arrow
    colunas = schema.names

    # Acumuladores de memória controlada
    total_linhas = 0
    null_counts = {col: 0 for col in colunas}

    # Para colunas numéricas (Float/Int), não rastreamos únicos exatos para não travar a RAM
    # Rastreamos apenas Min/Max para saber se é constante.
    numeric_stats = {col: {'min': float('inf'), 'max': float('-inf')} for col in colunas}

    # Para colunas categóricas/String, rastreamos únicos até o limite seguro
    unique_sets = {}
    unique_overflow = {}

    for col in colunas:
        field_type = schema.field(col).type
        # Se for浮点 (float) ou inteiro grande de valor contínuo, não usa set
        if 'float' in str(field_type).lower() or 'double' in str(field_type).lower():
            unique_sets[col] = None
        else:
            unique_sets[col] = set()
        unique_overflow[col] = False

    # ------------------------------------------------------------------------------
    # LOOP DE BLOCOS (CHUNKING)
    # ------------------------------------------------------------------------------
    try:
        for batch in pf.iter_batches(batch_size=TAMANHO_BLOCO):
            df = pl.from_arrow(batch)
            batch_height = df.height
            total_linhas += batch_height

            # 1. Contar Nulos do bloco
            batch_nulls = df.null_count().row(0)
            for i, col in enumerate(colunas):
                null_counts[col] += batch_nulls[i]

            # 2. Coletar Estatísticas e Únicos do bloco
            for col in colunas:
                serie = df[col]

                # Stats para numéricos
                if unique_sets[col] is None:
                    try:
                        # Min e Max rápidos
                        stats = serie.min_max()
                        if stats['min'] is not None and stats['min'] < numeric_stats[col]['min']:
                            numeric_stats[col]['min'] = stats['min']
                        if stats['max'] is not None and stats['max'] > numeric_stats[col]['max']:
                            numeric_stats[col]['max'] = stats['max']
                    except Exception:
                        pass

                # Únicos para categóricos (com safeguard de RAM)
                elif not unique_overflow[col]:
                    try:
                        # Pega únicos do bloco (muito rápido no Polars)
                        batch_uniques = serie.drop_nulls().unique().to_list()
                        current_set = unique_sets[col]

                        # Safeguard: Se estourar o limite, libera a memória e marca overflow
                        if len(current_set) + len(batch_uniques) > MAX_UNICOS_TRACK:
                            unique_overflow[col] = True
                            unique_sets[col] = None  # Libera RAM imediatamente
                            gc.collect()
                        else:
                            current_set.update(batch_uniques)
                    except Exception:
                        unique_overflow[col] = True
                        unique_sets[col] = None
                        gc.collect()

            # Limpeza agressiva do bloco
            del df
            del batch
            gc.collect()

    finally:
        del pf
        gc.collect()

    # ------------------------------------------------------------------------------
    # GERAÇÃO DO RELATÓRIO (DATAFRAME FINAL)
    # ------------------------------------------------------------------------------
    print(f"📊 Gerando relatório para {total_linhas:,} linhas...")

    resultados = []
    for col in colunas:
        pct_nulos = (null_counts[col] / total_linhas) * 100 if total_linhas > 0 else 0

        # Definir cardinalidade (valores únicos)
        if unique_sets[col] is None:
            if unique_overflow[col]:
                n_unicos = f"> {MAX_UNICOS_TRACK:,}"
                n_unicos_int = MAX_UNICOS_TRACK + 1
            else:
                # Era numérico, usamos min/max para checar se é constante
                min_val = numeric_stats[col]['min']
                max_val = numeric_stats[col]['max']
                if min_val == max_val and min_val != float('inf'):
                    n_unicos = "1 (Constante)"
                    n_unicos_int = 1
                else:
                    n_unicos = "Numérico Contínuo"
                    n_unicos_int = MAX_UNICOS_TRACK + 1 # Considera alta cardinalidade
        else:
            n_unicos_int = len(unique_sets[col])
            n_unicos = f"{n_unicos_int:,}"

        # Libera memória dos sets conforme processa o relatório
        unique_sets[col] = None

        # LÓGICA DE CLASSIFICAÇÃO (BASEADA NO TCC)
        status = "✅ MANTER (Feature Útil)"
        motivo = "Coluna com variância e preenchimento adequados."

        if col in COLUNAS_VAZAMENTO_DADOS:
            status = "❌ REMOVER (Data Leakage)"
            motivo = "Vazamento de dados (ex: produtividade só se conhece no fim da safra)."
        elif col in COLUNAS_IDENTIFICADORES:
            status = "❌ REMOVER (Identificador)"
            motivo = "Chave primária/ID. Não generaliza para novos pedidos."
        elif col in COLUNAS_REDUNDANTES_GEO:
            status = "❌ REMOVER (Redundante)"
            motivo = "Informação geográfica duplicada (já temos CD_IBGE_MUNICIPIO)."
        elif n_unicos_int == 1:
            status = "❌ REMOVER (Constante)"
            motivo = "A coluna possui apenas 1 valor único em todo o dataset."
        elif n_unicos_int >= total_linhas * 0.95:
            status = "❌ REMOVER (Alta Cardinalidade/ID)"
            motivo = "Quase todos os valores são únicos. Provável ID disfarçado."
        elif pct_nulos > 50.0:
            status = "⚠️ AVALIAR REMOÇÃO (Muitos Nulos)"
            motivo = f"Mais de 50% de dados faltantes ({pct_nulos:.1f}%). Introduz ruído."

        resultados.append({
            'Arquivo': nome_base,
            'Coluna': col,
            'Status': status,
            'Motivo': motivo,
            'Valores_Unicos': n_unicos,
            'Pct_Nulos': round(pct_nulos, 2)
        })

    gc.collect()
    return pl.DataFrame(resultados)


# ==============================================================================
# EXECUÇÃO PRINCIPAL
# ==============================================================================
if __name__ == "__main__":
    arquivos = sorted(glob.glob(os.path.join(PASTA_PARQUETS, "*.parquet")))

    if not arquivos:
        print(f"⚠️ Nenhum arquivo parquet encontrado em: {PASTA_PARQUETS}")
    else:
        print(f"🚀 Iniciando auditoria de {len(arquivos)} arquivos gigantes...")

        todos_relatorios = []

        for idx, arq in enumerate(arquivos, 1):
            try:
                df_relatorio = analisar_saude_arquivo(arq)
                todos_relatorios.append(df_relatorio)
                print(f"[{idx}/{len(arquivos)}] ✅ Concluído.")
            except Exception as e:
                print(f"[{idx}/{len(arquivos)}] ❌ Erro ao processar {arq}: {e}")

        if todos_relatorios:
            # Concatena tudo num único DataFrame Polars (muito mais leve que Pandas)
            df_final = pl.concat(todos_relatorios)

            # Ordena para mostrar os problemas primeiro
            ordem_status = ['❌ REMOVER (Data Leakage)', '❌ REMOVER (Identificador)',
                            '❌ REMOVER (Redundante)', '❌ REMOVER (Constante)',
                            '❌ REMOVER (Alta Cardinalidade/ID)',
                            '⚠️ AVALIAR REMOÇÃO (Muitos Nulos)', '✅ MANTER (Feature Útil)']

            # Cria coluna de ordem para sort
            mapeamento_ordem = {s: i for i, s in enumerate(ordem_status)}
            df_final = df_final.with_columns(
                pl.col('Status').replace(mapeamento_ordem).alias('ordem_sort')
            ).sort('ordem_sort').drop('ordem_sort')

            # Salva em CSV
            caminho_saida_csv = os.path.join(PASTA_RELATORIOS, "auditoria_colunas_sicor.csv")
            df_final.write_csv(caminho_saida_csv)

            # Salva também em Parquet (para você carregar rápido depois se quiser)
            caminho_saida_pq = os.path.join(PASTA_RELATORIOS, "auditoria_colunas_sicor.parquet")
            df_final.write_parquet(caminho_saida_pq)

            print("\n" + "="*60)
            print("🎉 AUDITORIA FINALIZADA!")
            print(f"📄 Relatório CSV salvo em: {caminho_saida_csv}")
            print(f"📦 Relatório Parquet salvo em: {caminho_saida_pq}")
            print("="*60)

            # Print de resumo rápido no console
            resumo = df_final.group_by('Status').len().sort('len', descending=True)
            print("\n📊 RESUMO GERAL DAS COLUNAS:")
            print(resumo)

🚀 Iniciando auditoria de 8 arquivos gigantes...

🔍 Analisando: full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet
[1/8] ❌ Erro ao processar /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Enriquecido_Completo/full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet: Parquet file size is 0 bytes

🔍 Analisando: full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2019.parquet
📊 Gerando relatório para 1,860,900 linhas...
[2/8] ✅ Concluído.

🔍 Analisando: full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2020.parquet
📊 Gerando relatório para 1,924,778 linhas...
[3/8] ✅ Concluído.

🔍 Analisando: full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2021.parquet
📊 Gerando relatório para 1,989,446 linhas...
[4/8] ✅ Concluído.

🔍 Analisando: full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2022.parquet
📊 Gerando relatório para 1,990,945 linhas...
[5/8] ✅ Concluído.

🔍 Analisando: full_final_enriquecido_SICOR_OPERACA

# Removendo Colunas Não Necessárias Para a IA

In [ ]:
# ==============================================================================
# LIMPEZA E SELEÇÃO DE FEATURES - OTIMIZADO (LEITURA SELETIVA NO DISCO)
# ==============================================================================
# Em vez de carregar tudo na RAM e depois deletar, este script usa o PyArrow
# para ler APENAS as colunas úteis diretamente do Parquet. As colunas proibidas
# (Data Leakage, IDs, Redundantes, Nulos) são ignoradas no nível do disco (I/O).
# Isso economiza RAM e é ordens de magnitude mais rápido.

import os
import gc
import glob
import pyarrow.parquet as pq
import polars as pl

# ------------------------------------------------------------------------------
# Variáveis de ambiente (Controle de Memória)
# ------------------------------------------------------------------------------
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["POLARS_MAX_THREADS"] = "2"
os.environ["POLARS_VERBOSE"] = "0"

# ==============================================================================
# CONFIGURAÇÕES DE CAMINHOS (ATUALIZADO PARA A VERSÃO MAIS RECENTE - DIA 17)
# ==============================================================================
# 🟢 PASTA DE ENTRADA: Apontando para a sua versão mais nova (2019-2025)
PASTA_PARQUETS_ORIGINAIS = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"
    "Parquet_Enriquecido_Completo"
)

# 🟢 PASTA DE SAÍDA: Onde os Parquets LIMPOS (apenas features úteis) serão salvos
PASTA_PARQUETS_LIMPOS = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"
    "Parquet_Limpo_Features_TCC"
)
os.makedirs(PASTA_PARQUETS_LIMPOS, exist_ok=True)

# Caminho do seu relatório de auditoria (gerado pelo script anterior com o recorte 19-25)
CAMINHO_CSV_AUDITORIA = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Relatorios_Saude_Colunas/auditoria_colunas_sicor.csv"
)

# Tamanho do bloco. Como agora não carregamos colunas inúteis,
# podemos usar blocos maiores (50k) sem medo de OOM (Out of Memory).
TAMANHO_BLOCO = 50_000

# ==============================================================================
# 1. MONTAGEM AUTOMÁTICA DA BLACKLIST (BASEADA NO SEU CSV DE AUDITORIA)
# ==============================================================================
def montar_blacklist_auditoria(caminho_csv: str) -> set:
    """
    Lê o CSV de auditoria e extrai todas as colunas marcadas para remoção
    ou avaliação crítica (nulos > 50%), garantindo o alinhamento com o TCC.
    """
    print("📖 Lendo relatório de auditoria para montar a Blacklist...")
    df_audit = pl.read_csv(caminho_csv)

    # Filtra colunas que o TCC determinou remover (❌) ou avaliar criticamente (⚠️)
    cols_para_descartar = df_audit.filter(
        pl.col('Status').str.contains('REMOVER') |
        pl.col('Status').str.contains('AVALIAR REMOÇÃO')
    )['Coluna'].unique().to_list()

    blacklist = set(cols_para_descartar)
    print(f"🚫 Blacklist montada com sucesso: {len(blacklist)} colunas serão descartadas.")

    # Exibe algumas colunas chave que garantimos que estão na blacklist
    exemplos = ['VL_PRODUTIV_OBTIDA', 'CNPJ_IF', 'NOME_MUNICIPIO', 'VL_AREA_INFORMADA']
    for ex in exemplos:
        status = "✅ NA BLACKLIST" if ex in blacklist else "⚠️ ATENÇÃO: NÃO ENCONTRADA"
        print(f"   -> {ex}: {status}")

    return blacklist

# ==============================================================================
# 2. FUNÇÃO DE PROCESSAMENTO (LEITURA SELETIVA E ESCRITA EM BLOCOS)
# ==============================================================================
def limpar_arquivo_parquet(caminho_origem: str, caminho_saida: str, blacklist: set):
    """
    Lê o Parquet original, ignora as colunas da blacklist no disco,
    e salva um novo Parquet apenas com as features válidas para o TCC.
    """
    nome_base = os.path.basename(caminho_origem)

    # Abre o arquivo original
    pf = pq.ParquetFile(caminho_origem)
    schema_orig = pf.schema_arrow
    colunas_originais = schema_orig.names

    # Filtra as colunas que DEVEM ser lidas (Interseção: Schema Original - Blacklist)
    colunas_para_ler = [c for c in colunas_originais if c not in blacklist]

    if not colunas_para_ler:
        print(f"⚠️ {nome_base}: Nenhuma coluna útil encontrada. Pulando...")
        return 0, 0

    # Lê o primeiro batch (1 linha) apenas para descobrir o schema de saída
    # Isso é crucial para o ParquetWriter saber a estrutura do novo arquivo
    first_batch = next(pf.iter_batches(batch_size=1, columns=colunas_para_ler))
    schema_saida = first_batch.schema

    # Inicializa o Writer do PyArrow (escrita em disco sem carregar tudo na RAM)
    # Usamos compressão 'snappy' que é o padrão ouro para equilíbrio CPU/Tamanho
    writer = pq.ParquetWriter(caminho_saida, schema_saida, compression='snappy')

    total_linhas_escritas = 0

    # Loop de blocos (Chunking)
    try:
        for batch in pf.iter_batches(batch_size=TAMANHO_BLOCO, columns=colunas_para_ler):
            writer.write_batch(batch)
            total_linhas_escritas += batch.num_rows

            # Limpeza de memória do batch atual
            del batch
            gc.collect()
    finally:
        writer.close()
        del pf
        gc.collect()

    return len(colunas_originais), len(colunas_para_ler)

# ==============================================================================
# EXECUÇÃO PRINCIPAL
# ==============================================================================
if __name__ == "__main__":
    # Monta a blacklist dinamicamente a partir do seu CSV de auditoria
    if not os.path.exists(CAMINHO_CSV_AUDITORIA):
        raise FileNotFoundError(f"❌ CSV de auditoria não encontrado em: {CAMINHO_CSV_AUDITORIA}")

    BLACKLIST_TCC = montar_blacklist_auditoria(CAMINHO_CSV_AUDITORIA)

    # Pega apenas os arquivos de 2019 a 2025 que estão na pasta nova
    arquivos = sorted(glob.glob(os.path.join(PASTA_PARQUETS_ORIGINAIS, "*.parquet")))

    if not arquivos:
        print(f"⚠️ Nenhum arquivo parquet encontrado em: {PASTA_PARQUETS_ORIGINAIS}")
    else:
        print(f"\n🚀 Iniciando limpeza de {len(arquivos)} arquivos (Recorte 2019-2025)...")
        print(f"📂 Salvando arquivos limpos em: {PASTA_PARQUETS_LIMPOS}\n")
        print("-" * 60)

        for idx, arq in enumerate(arquivos, 1):
            nome_base = os.path.basename(arq)
            caminho_saida = os.path.join(PASTA_PARQUETS_LIMPOS, nome_base)

            try:
                cols_orig, cols_final = limpar_arquivo_parquet(arq, caminho_saida, BLACKLIST_TCC)

                print(f"[{idx}/{len(arquivos)}] ✅ {nome_base}")
                print(f"   📉 Colunas: {cols_orig} -> {cols_final} (Descartadas: {cols_orig - cols_final})")

            except Exception as e:
                print(f"[{idx}/{len(arquivos)}] ❌ Erro ao processar {nome_base}: {e}")

        print("-" * 60)
        print("🎉 LIMPEZA FINALIZADA!")
        print(f"📦 Seus dados prontos para a esteira de IA estão em: {PASTA_PARQUETS_LIMPOS}")
        print("💡 Dica: Agora você pode usar esses arquivos limpos para gerar o dataset sintético de 5.000 produtores.")

📖 Lendo relatório de auditoria para montar a Blacklist...
🚫 Blacklist montada com sucesso: 40 colunas serão descartadas.
   -> VL_PRODUTIV_OBTIDA: ✅ NA BLACKLIST
   -> CNPJ_IF: ✅ NA BLACKLIST
   -> NOME_MUNICIPIO: ✅ NA BLACKLIST
   -> VL_AREA_INFORMADA: ✅ NA BLACKLIST

🚀 Iniciando limpeza de 8 arquivos (Recorte 2019-2025)...
📂 Salvando arquivos limpos em: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Limpo_Features_TCC

------------------------------------------------------------
[1/8] ❌ Erro ao processar full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2013.parquet: Parquet file size is 0 bytes
[2/8] ✅ full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2019.parquet
   📉 Colunas: 89 -> 49 (Descartadas: 40)
[3/8] ✅ full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2020.parquet
   📉 Colunas: 89 -> 49 (Descartadas: 40)
[4/8] ✅ full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2021.parquet
   📉 Colunas: 89 -> 49 (Descartadas:

# Verificando 100 linhas de todas as colunas dos Parquet

In [ ]:
# ==============================================================================
# EXTRAÇÃO DE AMOSTRA (100 LINHAS / TODAS AS COLUNAS) - OTIMIZADO
# ==============================================================================
# Este script lê APENAS as primeiras 100 linhas de um arquivo Parquet,
# mantendo TODAS as colunas, e salva em um CSV para análise manual no Excel/Sheets.

import os
import glob
import polars as pl

# ==============================================================================
# CONFIGURAÇÕES DE CAMINHOS
# ==============================================================================
# 🟢 PASTA DE ENTRADA: Onde estão os seus Parquets
PASTA_PARQUETS = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "Microdados do Crédito Rural e do Proagro - Sicor - Parquet/"
    "Parquet_Limpo_Features_TCC" # Altere para 'Parquet_Limpo_Features_TCC' se preferir a versão limpa
)

# 🟢 ARQUIVO ALVO:
# Por padrão, pegaremos o PRIMEIRO arquivo da pasta (ordenado alfabeticamente).
# Se quiser um ano específico, mude para o nome exato, ex: "sicor_2023.parquet"
ARQUIVO_ALVO = "primeiro_disponivel"

# 🟢 CAMINHO DE SAÍDA: Onde o CSV será salvo no seu Drive
CAMINHO_SAIDA_CSV = (
    "/content/drive/MyDrive/TCC - Ideias/"
    "amostra_100_linhas_todas_colunas.csv"
)

# ==============================================================================
# EXECUÇÃO
# ==============================================================================
def extrair_amostra(pasta, alvo, saida_csv):
    print(f"🔍 Buscando arquivos em: {pasta}")

    arquivos = sorted(glob.glob(os.path.join(pasta, "*.parquet")))

    if not arquivos:
        print(f"⚠️ Nenhum arquivo parquet encontrado na pasta.")
        return

    # Define qual arquivo ler
    if alvo == "primeiro_disponivel":
        caminho_arquivo = arquivos[0]
    else:
        caminho_arquivo = os.path.join(pasta, alvo)
        if not os.path.exists(caminho_arquivo):
            print(f"❌ Arquivo específico não encontrado: {caminho_arquivo}")
            return

    nome_arquivo = os.path.basename(caminho_arquivo)
    print(f"📂 Arquivo selecionado: {nome_arquivo}")
    print("⏳ Extraindo as primeiras 100 linhas (com todas as colunas)...")

    try:
        # A mágica do n_rows: O Polars instrui o PyArrow a ler APENAS os primeiros blocos
        # do disco necessários para preencher 100 linhas, ignorando os gigabytes restantes.
        df_amostra = pl.read_parquet(caminho_arquivo, n_rows=100)

        print(f"✅ Leitura concluída. DataFrame carregado: {df_amostra.shape[0]} linhas x {df_amostra.shape[1]} colunas.")

        # Salvar em CSV
        print(f"💾 Salvando CSV em: {saida_csv}")
        df_amostra.write_csv(saida_csv)

        print("🎉 Amostra gerada com sucesso! Você já pode abrir o CSV.")

        # Exibir um resumo no console do Colab para facilitar
        print("\n" + "="*60)
        print("📋 RESUMO DA AMOSTRA")
        print("="*60)
        print(f"📊 Linhas: {df_amostra.shape[0]} | Colunas: {df_amostra.shape[1]}")
        print("📝 Lista de Colunas:")
        for i, col in enumerate(df_amostra.columns, 1):
            print(f"   {i:02}. {col}")

    except Exception as e:
        print(f"❌ Ocorreu um erro durante a extração: {e}")

if __name__ == "__main__":
    # Descomente a linha abaixo se não tiver montado o Drive ainda na sua sessão
    # from google.colab import drive; drive.mount('/content/drive')

    extrair_amostra(PASTA_PARQUETS, ARQUIVO_ALVO, CAMINHO_SAIDA_CSV)

🔍 Buscando arquivos em: /content/drive/MyDrive/TCC - Ideias/Microdados do Crédito Rural e do Proagro - Sicor - Parquet/Parquet_Limpo_Features_TCC
📂 Arquivo selecionado: full_final_enriquecido_SICOR_OPERACAO_BASICA_ESTADO_2019.parquet
⏳ Extraindo as primeiras 100 linhas (com todas as colunas)...
✅ Leitura concluída. DataFrame carregado: 100 linhas x 49 colunas.
💾 Salvando CSV em: /content/drive/MyDrive/TCC - Ideias/amostra_100_linhas_todas_colunas.csv
🎉 Amostra gerada com sucesso! Você já pode abrir o CSV.

📋 RESUMO DA AMOSTRA
📊 Linhas: 100 | Colunas: 49
📝 Lista de Colunas:
   01. DT_EMISSAO
   02. DT_VENCIMENTO
   03. CD_CATEG_EMITENTE
   04. CD_FONTE_RECURSO
   05. CD_ESTADO
   06. CD_TIPO_SEGURO
   07. CD_PROGRAMA
   08. CD_TIPO_IRRIGACAO
   09. CD_TIPO_AGRICULTURA
   10. CD_FASE_CICLO_PRODUCAO
   11. CD_TIPO_CULTIVO
   12. CD_TIPO_INTGR_CONSOR
   13. CD_TIPO_GRAO_SEMENTE
   14. VL_JUROS
   15. VL_QUANTIDADE
   16. VL_PARC_CREDITO
   17. VL_PERC_RISCO_STN
   18. VL_PERC_RISCO_FUNDO_C

# Explorando Dados Arquivos KML